# NEWv4 rank 58: classical post-processing, R4/R5 only

This standalone notebook extends the latest NEWv3 classical implementation to five
teleportation links. It prepares/decodes the disjoint AME registers before the
disjoint Bell-measurement layers; no physical conditional Pauli gates are applied.
Each checkpoint still starts from zero and is compiled/scheduled independently.

Only R4 and R5 reference–receiver tomography is acquired, for channels A and B.
Earlier links and their Bell measurements are part of each circuit, but there are
no R0–R3 endpoint acquisition jobs. Pauli-frame correction XORs all outcomes from
rounds 1 through the measured checkpoint; M3 preserves the full joint distribution.

Maximum allocation: 29 logical qubits (four input/reference qubits plus five
five-qubit AME registers), with a common initial layout for all checkpoints.
Current source trial, shot, DD, optimisation-level and calibration settings are
retained. RUN_EXPERIMENT defaults to False; explicitly enable it to submit jobs.
Raw and M3 rho_real/rho_imag are retained. At R4, input_rho is marked unavailable
because R3 is not measured. At R5, the earlier checkpoint matrices are independent
experimental references, not live quantum states reused between jobs.

Only original-style PNG fidelity bar plots are generated; no PDFs, CHSH,
feed-forward acquisition, noisy simulation, heatmaps or line plots.
Data: mcm_shared_prep_rank58_classical_R456/<timestamp>/classical/.
Plots: epr_replots/<timestamp>/classical/. Existing notebooks are unchanged.


## Imports and configuration
Start with a fresh kernel. Your current acquisition settings are preserved.
Review `RUN_EXPERIMENT` before running the acquisition cell; replotting saved
records does not require enabling it.

In [ ]:
import copy
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t
from scipy.linalg import hadamard

from qiskit import (
    QuantumCircuit, QuantumRegister, ClassicalRegister, transpile,
    generate_preset_pass_manager,
)
from qiskit.circuit.equivalence_library import (
    EquivalenceLibrary, SessionEquivalenceLibrary as sel,
)
from qiskit.circuit.library import XGate, YGate
from qiskit.quantum_info import Statevector
from qiskit.primitives.containers import BitArray, SamplerPub
from qiskit.circuit.controlflow import ControlFlowOp
from qiskit.transpiler import InstructionProperties, PassManager
from qiskit.transpiler.passes import BasisTranslator
from qiskit_experiments.framework import ExperimentData
from qiskit_experiments.library import StateTomography
from contextlib import ExitStack
import atexit
from qiskit_ibm_runtime import (
    QiskitRuntimeService, SamplerV2 as Sampler, Session,
)
from qiskit_ibm_runtime.circuit import MidCircuitMeasure
from qiskit_ibm_runtime.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis, PadDelay, PadDynamicalDecoupling,
)

# A hardware execution cell gets one dedicated Runtime Session.  ExitStack
# enters the Session context, and the IPython post-run hook exits it even if
# the cell raises.  All hardware samplers created in that cell share it.
_HARDWARE_SESSION_STACK = None
_HARDWARE_SESSIONS = {}
_HARDWARE_SESSION_CLEANUP_REGISTERED = False


def close_hardware_sessions(_result=None):
    """Close this cell's Runtime sessions and release dedicated access."""
    global _HARDWARE_SESSION_STACK
    global _HARDWARE_SESSIONS
    global _HARDWARE_SESSION_CLEANUP_REGISTERED
    stack = _HARDWARE_SESSION_STACK
    _HARDWARE_SESSION_STACK = None
    _HARDWARE_SESSIONS = {}
    close_error = None
    if stack is not None:
        try:
            stack.close()
        except Exception as exc:
            close_error = exc
    if _HARDWARE_SESSION_CLEANUP_REGISTERED:
        try:
            shell = get_ipython()
            if shell is not None:
                shell.events.unregister(
                    "post_run_cell", close_hardware_sessions
                )
        except (NameError, ValueError):
            pass
        _HARDWARE_SESSION_CLEANUP_REGISTERED = False
    if close_error is not None:
        raise close_error


def make_hardware_session_sampler(backend):
    """Return SamplerV2 in a shared, cell-scoped Runtime Session."""
    global _HARDWARE_SESSION_STACK
    global _HARDWARE_SESSION_CLEANUP_REGISTERED
    if _HARDWARE_SESSION_STACK is None:
        _HARDWARE_SESSION_STACK = ExitStack()
        try:
            shell = get_ipython()
        except NameError:
            shell = None
        if shell is not None:
            shell.events.register(
                "post_run_cell", close_hardware_sessions
            )
            _HARDWARE_SESSION_CLEANUP_REGISTERED = True
        else:
            atexit.register(close_hardware_sessions)
    backend_name = getattr(backend, "name", str(backend))
    if callable(backend_name):
        backend_name = backend_name()
    if backend_name not in _HARDWARE_SESSIONS:
        _HARDWARE_SESSIONS[backend_name] = (
            _HARDWARE_SESSION_STACK.enter_context(
                Session(backend=backend)
            )
        )
        print(
            f"Dedicated IBM Runtime Session opened on {backend_name}; "
            "it will close when this execution cell finishes.",
            flush=True,
        )
    session = _HARDWARE_SESSIONS[backend_name]
    return Sampler(mode=session)
import mthree
MTHREE_IMPLEMENTATION = "mthree.M3Mitigation"
import mapomatic as mm

print("Imports complete.")

# ---------------------------- Configuration ----------------------------
EXPECTED_RUNTIME_VERSION = "0.43.0"
EXPERIMENT_VARIANT = 'Classical-only overlap-enabled MCM cascade; R4/R5'
BACKEND_NAME = "ibm_pittsburgh"
# DISENTANGLER_RANK = 2  # moved up from below so OUTPUT_DIR can be tagged by rank
DISENTANGLER_RANK = 58  # Former V234 choice (source q0->q4, source q1->q3, leftover q2);
                        # physical_rank 58/60 in synthesize_W012_dagger_same_AME.ipynb
                        # (3 two-qubit, 13 single-qubit gates, depth 10 -- worse than rank-2
                        # by design, for comparison. OUTPUT_DIR below picks up rank58/ automatically.
N_TRIALS = 1  # Independent acquisitions per level/channel/round/DD policy.
# Balanced Mapomatic limits for every A/B state-analysis pair.
# Pair risk = 1 - (1-readout_error_left)*(1-readout_error_right).
LAYOUT_MAX_SOURCE_PAIR_READOUT_RISK = 0.12
LAYOUT_MAX_ANALYSIS_PAIR_READOUT_RISK = 0.18
LAYOUT_MAX_CHANNEL_READOUT_RISK_GAP = 0.06
LAYOUT_REPICK_ATTEMPTS = 8
LAYOUT_MEAN_READOUT_WEIGHT = 1.0
LAYOUT_WORST_READOUT_WEIGHT = 2.0
LAYOUT_CHANNEL_BALANCE_WEIGHT = 1.0
SHOTS = 5000
M3_CALIBRATION_SHOTS = 5000
OPTIMIZATION_LEVEL = 2  # MINIMAL VERIFICATION RUN: was 3 - testing whether a lower transpiler level changes round-2/3 XY4 behavior
SEED_TRANSPILER = 42

# All files are written here. Change this before execution if desired.
OUTPUT_ROOT = Path.cwd()
OUTPUT_DIR = OUTPUT_ROOT / f"mcm_shared_prep_rank{DISENTANGLER_RANK}_classical_R456"
HARDWARE_CALIBRATION_HISTORY_PATH = Path.cwd() / 'hardware_calibration_history.txt'  # shared across all four notebooks; append-only

MCM_DD_AUDIT_CSV = OUTPUT_DIR / "mcm_dynamic_dd_schedule_audit.csv"

# Safety guard: set True only when intentionally submitting hardware jobs.
RUN_EXPERIMENT = False  # Set True intentionally to submit QPU jobs.
SHOW_SCHEDULER_TIMING = False  # Scheduler plotting disabled by request.
MAX_TIMING_PLOTS = 12
STATUS_POLL_SECONDS = 15
MAINTENANCE_POLL_SECONDS = 600  # Recheck backend availability every 10 min.
RUNTIME_INSTANCE = None  # Search all accessible projects for BACKEND_NAME.
JOB_MAX_ATTEMPTS = 5
JOB_RETRY_BASE_SECONDS = 30
RETRYABLE_IBM_ERROR_CODES = (
    {
        # IBM's error registry explicitly recommends retrying these
        # availability, queue-publication, rate-limit, timeout, internal,
        # and job-state synchronization failures.
        1004, 1011, 1202, 1245, 1302, 1307, 1319, 1323,
        1808, 1809, 3202, 3203, 3208, 3212, 3213, 3214,
        3245, 3246, 3259, 3704, 3900, 3911,
        4300, 4301, 4303, 4305, 4308, 4314,
        5203, 6072, 9999,
    }
    | set(range(6051, 6069))
    | set(range(9700, 9711))
)
BACKEND_PENDING_WARNING_THRESHOLD = 500
ENABLE_DD = False
DD_SEQUENCE = "XpXm"
DD_SCHEDULING_METHOD = "alap"
DD_POLICIES = ["none", "XY4", "XX"]
TIMING_PLOTS_PER_POLICY = 4



print(f"RUN_EXPERIMENT={RUN_EXPERIMENT}")
print(f"Outputs: {OUTPUT_DIR.resolve()}")


def require_runtime_043():
    """Keep the supplied hardware Runtime version requirement unchanged."""
    try:
        installed = version("qiskit-ibm-runtime")
    except PackageNotFoundError as exc:
        raise RuntimeError("qiskit-ibm-runtime is not installed.") from exc
    if installed != EXPECTED_RUNTIME_VERSION:
        raise RuntimeError(f"Hardware execution requires qiskit-ibm-runtime=={EXPECTED_RUNTIME_VERSION}; found {installed}. "
                           "Definition/plot cells and offline checks do not require this version.")

    # The Sampler's built-in DD option remains disabled for MCM. This sweep
    # uses Runtime 0.43's dynamic-circuit scheduling passes before submission.

from qiskit_ibm_runtime.circuit import MidCircuitMeasure
from qiskit.circuit import Measure
from qiskit import transpile as _qiskit_transpile
from qiskit_experiments.framework import BaseExperiment


def _append_measure_2(circuit, qubits, clbits):
    """Broadcast IBM readout to scalar bits, registers, or bit lists."""
    qubits = circuit._qbit_argument_conversion(qubits)
    clbits = circuit._cbit_argument_conversion(clbits)
    if len(qubits) != len(clbits):
        raise ValueError('Readout requires one classical bit per measured qubit.')
    for qubit, clbit in zip(qubits, clbits):
        circuit.append(MidCircuitMeasure(), [qubit], [clbit])
    return circuit


def _readout_is_simulator(backend):
    if backend is None:
        return False
    try:
        return bool(backend.configuration().simulator)
    except (AttributeError, TypeError):
        return 'aer' in type(backend).__module__.lower()


def _rewrite_readouts(circuit, simulator=False):
    """Preserve registers/layout/metadata and recursively use one readout kind."""
    result = circuit.copy_empty_like()
    for item in circuit.data:
        operation = item.operation
        if operation.name in ('measure', 'measure_2'):
            operation = Measure() if simulator else MidCircuitMeasure()
        elif getattr(operation, 'blocks', ()):
            operation = operation.replace_blocks([
                _rewrite_readouts(block, simulator) for block in operation.blocks
            ])
        result.append(operation, item.qubits, item.clbits)
    result._layout = circuit.layout
    return result


def _validate_physical_readouts(circuit, backend, root_indices=None):
    if backend is None or _readout_is_simulator(backend):
        return
    root_indices = list(range(circuit.num_qubits)) if root_indices is None else root_indices
    for item in circuit.data:
        physical = [root_indices[circuit.find_bit(q).index] for q in item.qubits]
        if item.operation.name in ('measure', 'measure_2'):
            if item.operation.name != 'measure_2':
                raise RuntimeError('Hardware circuit contains standard measure.')
            if not backend.target.instruction_supported('measure_2', tuple(physical)):
                raise RuntimeError(f'measure_2 is unsupported on physical qubits {physical}.')
        for block in getattr(item.operation, 'blocks', ()):
            _validate_physical_readouts(block, backend, physical)


def transpile(circuits, backend=None, *args, **kwargs):
    """Normalize before routing/scheduling; validate every measured physical qubit."""
    single = isinstance(circuits, QuantumCircuit)
    prepared = [_rewrite_readouts(c, _readout_is_simulator(backend))
                for c in ([circuits] if single else circuits)]
    result = _qiskit_transpile(prepared[0] if single else prepared, backend, *args, **kwargs)
    for circuit in ([result] if single else result):
        _validate_physical_readouts(circuit, backend)
    return result


# Qiskit Experiments generates its own terminal measurements (including
# LocalReadoutError/MST and RB). Normalize its routed circuits as well.
if not hasattr(BaseExperiment, '_original_readout_transpiled_circuits'):
    BaseExperiment._original_readout_transpiled_circuits = BaseExperiment._transpiled_circuits


def _experiment_measure_2_circuits(self):
    options = dict(self.transpile_options.__dict__)
    options['initial_layout'] = list(self.physical_qubits)
    return transpile(self.circuits(), self.backend, **options)


BaseExperiment._transpiled_circuits = _experiment_measure_2_circuits

if not hasattr(BaseExperiment, '_original_readout_run_jobs'):
    BaseExperiment._original_readout_run_jobs = BaseExperiment._run_jobs


def _experiment_measure_2_jobs(self, circuits, sampler=None, **run_options):
    # Covers experiments such as optimized RB with their own transpiler.
    circuits = [_rewrite_readouts(c, _readout_is_simulator(self.backend)) for c in circuits]
    for circuit in circuits:
        _validate_physical_readouts(circuit, self.backend)
    return self._original_readout_run_jobs(circuits, sampler=sampler, **run_options)


BaseExperiment._run_jobs = _experiment_measure_2_jobs


def _calibrate_all_measure_2(mitigator, qubits, shots=5000):
    """Local assignment matrices measured with the same hardware instruction."""
    backend = mitigator.system
    physical = list(dict.fromkeys(int(q) for q in qubits))
    circuits = []
    for prepared in (0, 1):
        circuit = QuantumCircuit(len(physical), len(physical))
        if prepared:
            circuit.x(range(len(physical)))
        _append_measure_2(circuit, range(len(physical)), range(len(physical)))
        circuits.append(circuit)
    compiled = transpile(circuits, backend, initial_layout=physical, optimization_level=0)
    sampler = (Sampler(mode=backend) if _readout_is_simulator(backend)
               else make_hardware_session_sampler(backend))
    result = sampler.run(compiled, shots=int(shots)).result()
    matrices = [None] * backend.num_qubits
    for logical, qubit in enumerate(physical):
        matrix = np.zeros((2, 2))
        for prepared, pub in enumerate(result):
            counts = pub.data.c.get_counts()
            for bits, count in counts.items():
                matrix[int(bits.replace(' ', '')[::-1][logical]), prepared] += count
        matrix /= matrix.sum(axis=0)
        matrices[qubit] = matrix
    mitigator.cals_from_matrices(matrices)
    return mitigator

OPT_LEVELS_TO_TEST = [2, 3]
OUTPUT_BASE_DIR = OUTPUT_ROOT / f"mcm_shared_prep_rank{DISENTANGLER_RANK}_classical_R456"
CHECKPOINT_ROUNDS = (4, 5)
MAX_ROUNDS = max(CHECKPOINT_ROUNDS)
N_LOGICAL_QUBITS = 4 + 5 * MAX_ROUNDS  # 29; no reset/reuse of AME registers.


## Rank-specific AME preparation, decoder and receiver identities


In [ ]:
RANK_ACTIVE_QUBITS = (2, 3, 4)


RANK_CHANNEL_RESOURCE_ROLES = (("A", 0, 4), ("B", 1, 3))


RANK_RECEIVERS = {"A": 4, "B": 3}


def _apply_ame52_prep(qc, q):
    """Append the AME(5,2) preparation used by v13."""
    qc.h(q[2]); qc.h(q[3])
    qc.cx(q[2], q[1])
    qc.cx(q[1], q[0])
    qc.h(q[2])
    qc.cx(q[2], q[4])
    qc.cx(q[3], q[2])
    qc.cx(q[2], q[0])
    return qc


def build_V234_dagger_current():
    """Exact synthesized decoder for the currently selected rank (local active-qubit order)."""
    # -- rank-2 decoder (kept for reference, superseded below) --
    # q = QuantumRegister(3, "q")
    # qc = QuantumCircuit(q, name="rank2_decoder")
    # qc.cx(q[1], q[2]); qc.h(q[2]); qc.s(q[2])
    # qc.cx(q[0], q[1]); qc.h(q[0]); qc.s(q[0])
    # qc.cx(q[0], q[2]); qc.sdg(q[2]); qc.h(q[2])
    # return qc
    #
    # Best-of-64 Bravyi-Maslov completion for the "Former V234 choice"
    # pairing (source q0->q4, source q1->q3, leftover q2), physical_rank
    # 58/60. Local wire order matches RANK_ACTIVE_QUBITS = (2, 3, 4):
    # local q[0]=AME q2, q[1]=AME q3, q[2]=AME q4.
    q = QuantumRegister(3, "q")
    qc = QuantumCircuit(q, name="v234_former_decoder")
    qc.h(q[0]); qc.x(q[1]); qc.x(q[2])
    qc.cx(q[1], q[2]); qc.sdg(q[2]); qc.h(q[2])
    qc.h(q[1]); qc.s(q[1])
    qc.cx(q[0], q[2]); qc.sdg(q[2]); qc.h(q[2])
    qc.h(q[0]); qc.s(q[0])
    qc.cx(q[1], q[2]); qc.sdg(q[2]); qc.h(q[2])
    return qc


def apply_V234_dagger(qc, *ame_qubits):
    """Apply the selected decoder to its rank-specific AME qubits."""
    if len(ame_qubits) == 1 and hasattr(ame_qubits[0], "__len__"):
        ame_qubits = tuple(ame_qubits[0])
    if len(ame_qubits) == 5:
        targets = [ame_qubits[index] for index in RANK_ACTIVE_QUBITS]
    elif len(ame_qubits) == 3:
        targets = list(ame_qubits)
    else:
        raise ValueError("Pass either the five AME qubits or the three active decoder qubits.")
    qc.append(build_V234_dagger_current().to_instruction(), targets)
    return qc


PHI_PLUS = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)


def metrics_from_density_matrix(rho):
    """Entanglement fidelity to Phi+; no extra measurement settings."""
    rho = np.asarray(rho, dtype=complex)
    rho = .5 * (rho + rho.conj().T)
    rho /= np.trace(rho)
    return float(np.real(np.vdot(PHI_PLUS, rho @ PHI_PLUS)))


CASCADE_SCHEME = 'MCM'


TRAJECTORY_STATE_PAIRS = {
    channel: {
        r: (f'ref_{channel}', f'in_{channel}' if r == 0 else f'q{RANK_RECEIVERS[channel]}_R{r}',
            (ref, initial if r == 0 else 4 + 5 * (r - 1) + RANK_RECEIVERS[channel]),
            f'rho(ref_{channel}, in_{channel})' if r == 0
            else f'rho(ref_{channel}, q{RANK_RECEIVERS[channel]}_R{r})')
        for r in range(MAX_ROUNDS + 1)
    }
    for channel, ref, initial in (('A', 0, 2), ('B', 1, 3))
}


def _state_role(round_number):
    r = int(round_number)
    if r == 0:
        return 'measured_input_state'
    if not 1 <= r <= MAX_ROUNDS:
        raise ValueError('Round outside the configured cascade.')
    return f'round_{r}_output_and_round_{r + 1}_input' if r < MAX_ROUNDS else f'round_{r}_output'



## Backend, readout calibration and job-recovery helpers

In [ ]:
def _save_m3_calibration_audit(payload, output_dir):
    """Write a uniquely named audit, including in long Windows directories."""
    import os
    from uuid import uuid4
    output_dir = Path(os.path.abspath(output_dir))
    audit_path = output_dir / f'm3_{uuid4().hex}.json'
    io_path = audit_path
    if os.name == 'nt':
        # Extended paths avoid dependence on the Windows LongPathsEnabled
        # setting. UNC paths require the extended UNC prefix.
        slash = chr(92)
        prefix = slash * 2 + '?' + slash
        text = str(audit_path)
        if not text.startswith(prefix):
            text = (prefix + 'UNC' + slash + text[2:]
                    if text.startswith(slash * 2) else prefix + text)
        io_path = Path(text)
    io_path.parent.mkdir(parents=True, exist_ok=True)
    with io_path.open('x', encoding='utf-8') as stream:
        json.dump(payload, stream, indent=2, allow_nan=False)
    return audit_path


class _LayoutUncalibrated(Exception):
    pass


def _symmetric_pair_gate_error_cost(circuit, layouts, backend):
    """Mapomatic cost robust to symmetric CZ ordering/missing couplers."""
    from qiskit_ibm_runtime.models.backend_properties import BackendPropertyError

    def two_qubit_error(properties, gate_name, p0, p1):
        try:
            return properties.gate_error(gate_name, [p0, p1])
        except BackendPropertyError:
            try:
                return properties.gate_error(gate_name, [p1, p0])
            except BackendPropertyError as exc:
                raise _LayoutUncalibrated(
                    f"No {gate_name} calibration for ({p0},{p1}) "
                    "in either ordering."
                ) from exc

    properties = backend.properties()
    scored = []
    skipped = 0
    for layout in layouts:
        fidelity_product = 1.0
        try:
            for item in circuit.data:
                operation, qargs = item.operation, item.qubits
                if operation.num_qubits == 2 and operation.name != "barrier":
                    q0 = circuit.find_bit(qargs[0]).index
                    q1 = circuit.find_bit(qargs[1]).index
                    fidelity_product *= 1.0 - two_qubit_error(
                        properties, operation.name, layout[q0], layout[q1]
                    )
                elif operation.name in ("sx", "x"):
                    q0 = circuit.find_bit(qargs[0]).index
                    fidelity_product *= 1.0 - properties.gate_error(
                        operation.name, layout[q0]
                    )
                elif operation.name in ("measure", "reset"):
                    q0 = circuit.find_bit(qargs[0]).index
                    fidelity_product *= 1.0 - properties.readout_error(
                        layout[q0]
                    )
        except _LayoutUncalibrated:
            skipped += 1
            continue
        scored.append((list(layout), 1.0 - fidelity_product))
    return scored, skipped


def select_physical_layout(circuit, backend, seed=SEED_TRANSPILER):
    """Select the full logical layout using v13's Mapomatic workflow.

    Measurements and dynamic single-qubit correction blocks are removed only
    from the temporary layout proxy because they add no coupling-map edges.
    The actual executed circuit remains unchanged. Candidate embeddings are
    ranked using the backend's current gate/readout calibration snapshot.
    """
    layout_proxy = QuantumCircuit(
        *circuit.qregs, name=f"{circuit.name}_mapomatic_proxy"
    )
    skipped_control = skipped_measurements = 0
    for item in circuit.data:
        operation, qargs, cargs = item.operation, item.qubits, item.clbits
        if isinstance(operation, ControlFlowOp) or operation.name in {
            "if_else", "while_loop", "for_loop", "switch_case"
        }:
            skipped_control += 1
            continue
        if operation.name == "measure":
            skipped_measurements += 1
            continue
        if cargs:
            continue
        layout_proxy.append(operation, qargs, [])

    routed = transpile(
        layout_proxy,
        backend=backend,
        optimization_level=OPTIMIZATION_LEVEL,
        seed_transpiler=int(seed),
    )
    compact = mm.deflate_circuit(routed)
    candidates = mm.matching_layouts(compact, backend)
    if not candidates:
        raise RuntimeError(
            f"Mapomatic found no {circuit.num_qubits}-qubit placement on "
            f"{backend.name}."
        )

    scored, calibration_skips = _symmetric_pair_gate_error_cost(
        compact, candidates, backend
    )
    scored.sort(key=lambda item: item[1])
    if not scored:
        raise RuntimeError(
            "Mapomatic found topological candidates, but none had complete "
            "calibration data. Retry after the next backend calibration."
        )
    best_layout, best_score = scored[0]

    # Routing can temporarily activate bridge qubits. Mapomatic ranks the
    # complete routed region, then this block retains only the original nine
    # logical qubits required by Qiskit's initial_layout argument.
    if len(best_layout) != circuit.num_qubits:
        active_qubits, _ = mm.circuits.active_bits(routed)
        active_sorted = sorted(
            active_qubits, key=lambda q: routed.find_bit(q).index
        )
        physical_to_compact = {
            routed.find_bit(q).index: index
            for index, q in enumerate(active_sorted)
        }
        original_physical = routed.layout.initial_index_layout(
            filter_ancillas=True
        )
        compact_indices = [
            physical_to_compact[physical] for physical in original_physical
        ]
        best_layout = [best_layout[index] for index in compact_indices]

    if len(best_layout) != circuit.num_qubits:
        raise RuntimeError(
            f"Mapomatic returned {len(best_layout)} entries for a "
            f"{circuit.num_qubits}-qubit circuit."
        )
    if len(set(best_layout)) != len(best_layout):
        raise RuntimeError(f"Mapomatic returned duplicate qubits: {best_layout}")

    print(
        f"Mapomatic selected {best_layout} on {backend.name}; "
        f"calibration score={best_score:.6f}; "
        f"candidates={len(scored)}, skipped={calibration_skips}; "
        f"proxy omitted {skipped_control} control-flow blocks and "
        f"{skipped_measurements} measurements."
    )
    return list(best_layout)


def select_balanced_physical_layout(circuit, backend, seed=SEED_TRANSPILER):
    """Pick one layout that is readout-safe for both channels and rounds.

    Repicking uses different transpiler seeds and is completed before any
    M3 calibration or experiment job is submitted.  A cascaded state cannot
    change layout between rounds, so all configured A/B receiver pairs are constrained
    simultaneously.
    """
    layout_proxy = QuantumCircuit(
        *circuit.qregs, name=f"{circuit.name}_balanced_mapomatic_proxy"
    )
    skipped_control = skipped_measurements = 0
    for item in circuit.data:
        operation, qargs, cargs = item.operation, item.qubits, item.clbits
        if isinstance(operation, ControlFlowOp) or operation.name in {
            "if_else", "while_loop", "for_loop", "switch_case"
        }:
            skipped_control += 1
            continue
        if operation.name == "measure":
            skipped_measurements += 1
            continue
        if cargs:
            continue
        layout_proxy.append(operation, qargs, [])

    # Rank-aware source/receiver pairs through the configured maximum round.
    n_rounds = min(MAX_ROUNDS, (circuit.num_qubits - 4) // 5)
    pair_specs = {
        f'{channel}_R{r}': (indices[0], indices[1], 'source' if r == 0 else 'analysis')
        for channel, pairs in TRAJECTORY_STATE_PAIRS.items()
        for r, (_, _, indices, _) in pairs.items()
        if n_rounds >= 1 and r <= n_rounds
    }

    properties = backend.properties()
    excluded_readout = set()
    if properties is not None:
        excluded_readout.update(properties.faulty_qubits())
    print(f'Excluded readout qubits on {backend.name}: {sorted(excluded_readout)}')
    if properties is None:
        raise RuntimeError("Backend properties are unavailable for balanced layout selection.")

    accepted = {}
    best_rejected = None
    for attempt in range(int(LAYOUT_REPICK_ATTEMPTS)):
        attempt_seed = int(seed) + attempt
        routed = transpile(
            layout_proxy, backend=backend,
            optimization_level=OPTIMIZATION_LEVEL,
            seed_transpiler=attempt_seed,
        )
        compact = mm.deflate_circuit(routed)
        candidates = mm.matching_layouts(compact, backend)
        if not candidates:
            print(f"Balanced layout attempt {attempt + 1}/{LAYOUT_REPICK_ATTEMPTS}: no embeddings.")
            continue
        scored, calibration_skips = _symmetric_pair_gate_error_cost(
            compact, candidates, backend
        )
        active_qubits, _ = mm.circuits.active_bits(routed)
        active_sorted = sorted(
            active_qubits, key=lambda q: routed.find_bit(q).index
        )
        routed_to_compact = {
            routed.find_bit(q).index: index
            for index, q in enumerate(active_sorted)
        }
        original_routed = list(
            routed.layout.initial_index_layout(filter_ancillas=True)
        )
        if len(original_routed) != circuit.num_qubits:
            raise RuntimeError(
                "Could not map Mapomatic candidates back to every original logical qubit."
            )

        passed_this_attempt = 0
        for candidate, gate_score in scored:
            try:
                logical_layout = [
                    int(candidate[routed_to_compact[int(routed_index)]])
                    for routed_index in original_routed
                ]
                if excluded_readout.intersection(logical_layout):
                    continue
                readout = {}
                for physical in set(logical_layout):
                    instruction = backend.target['measure_2'].get((physical,))
                    error = getattr(instruction, 'error', None)
                    # Ordinary readout calibration is only a ranking proxy
                    # when IBM exposes no measure_2-specific error estimate.
                    if error is None:
                        error = properties.readout_error(physical)
                    readout[physical] = float(error)
            except Exception:
                continue
            if not all(
                    np.isfinite(value) and 0.0 <= value < 1.0
                    for value in readout.values()):
                continue
            if len(set(logical_layout)) != len(logical_layout):
                continue

            measured_indices = range(circuit.num_qubits)
            if any(not backend.target.instruction_supported('measure_2', (logical_layout[index],))
                   for index in measured_indices):
                continue
            pair_risks = {}
            for label, (left, right, _) in pair_specs.items():
                e_left = readout[logical_layout[left]]
                e_right = readout[logical_layout[right]]
                pair_risks[label] = 1.0 - (1.0 - e_left) * (1.0 - e_right)
            source_risks = [
                risk for label, risk in pair_risks.items() if label.endswith("R0")
            ] or [0.0]
            analysis_risks = [
                risk for label, risk in pair_risks.items() if not label.endswith("R0")
            ] or [0.0]
            a_risk = max(
                [risk for label, risk in pair_risks.items() if label.startswith("A_")] or [0.0]
            )
            b_risk = max(
                [risk for label, risk in pair_risks.items() if label.startswith("B_")] or [0.0]
            )
            worst_source = max(source_risks)
            worst_analysis = max(analysis_risks)
            channel_gap = abs(a_risk - b_risk)
            risks = list(pair_risks.values()) or [0.0]
            objective = (
                float(gate_score)
                + LAYOUT_MEAN_READOUT_WEIGHT * float(np.mean(risks))
                + LAYOUT_WORST_READOUT_WEIGHT * max(risks)
                + LAYOUT_CHANNEL_BALANCE_WEIGHT * channel_gap
            )
            passes = (
                worst_source <= LAYOUT_MAX_SOURCE_PAIR_READOUT_RISK
                and worst_analysis <= LAYOUT_MAX_ANALYSIS_PAIR_READOUT_RISK
                and channel_gap <= LAYOUT_MAX_CHANNEL_READOUT_RISK_GAP
            )
            candidate_info = {
                "layout": logical_layout, "gate_score": float(gate_score),
                "objective": float(objective), "pair_risks": pair_risks,
                "worst_source": worst_source,
                "worst_analysis": worst_analysis,
                "channel_gap": channel_gap, "seed": attempt_seed,
            }
            violation = max(
                worst_source / LAYOUT_MAX_SOURCE_PAIR_READOUT_RISK,
                worst_analysis / LAYOUT_MAX_ANALYSIS_PAIR_READOUT_RISK,
                channel_gap / LAYOUT_MAX_CHANNEL_READOUT_RISK_GAP,
            )
            if best_rejected is None or (violation, objective) < best_rejected[0]:
                best_rejected = ((violation, objective), candidate_info)
            if passes:
                passed_this_attempt += 1
                key = tuple(logical_layout)
                if key not in accepted or objective < accepted[key]["objective"]:
                    accepted[key] = candidate_info
        print(
            f"Balanced layout attempt {attempt + 1}/{LAYOUT_REPICK_ATTEMPTS}: "
            f"seed={attempt_seed}, candidates={len(scored)}, "
            f"passed={passed_this_attempt}, calibration-skipped={calibration_skips}."
        )

    if not accepted:
        detail = "no candidate had complete readout calibration"
        if best_rejected is not None:
            info = best_rejected[1]
            detail = (
                f"best rejected layout={info['layout']}, "
                f"source risk={info['worst_source']:.3f}, "
                f"analysis risk={info['worst_analysis']:.3f}, "
                f"A/B gap={info['channel_gap']:.3f}, "
                f"pair risks={info['pair_risks']}"
            )
        raise RuntimeError(
            f"No balanced Mapomatic layout passed after {LAYOUT_REPICK_ATTEMPTS} "
            f"automatic repicks; {detail}. Limits: source<="
            f"{LAYOUT_MAX_SOURCE_PAIR_READOUT_RISK:.3f}, analysis<="
            f"{LAYOUT_MAX_ANALYSIS_PAIR_READOUT_RISK:.3f}, A/B gap<="
            f"{LAYOUT_MAX_CHANNEL_READOUT_RISK_GAP:.3f}. No calibration or "
            "experiment job has been submitted."
        )

    best = min(accepted.values(), key=lambda item: item["objective"])
    risk_text = ", ".join(
        f"{label}={risk:.3f}" for label, risk in best["pair_risks"].items()
    )
    print(
        f"Balanced Mapomatic selected {best['layout']} on {backend.name}; "
        f"seed={best['seed']}, objective={best['objective']:.6f}, "
        f"gate score={best['gate_score']:.6f}; {risk_text}; "
        f"proxy omitted {skipped_control} control-flow blocks and "
        f"{skipped_measurements} measurements."
    )
    return list(best["layout"])


def _job_identifier(job):
    try:
        return str(job.job_id())
    except Exception:
        return "local/no-runtime-id"


def _job_status(job):
    try:
        status = job.status()
        return getattr(status, "name", str(status))
    except Exception:
        return "status unavailable"


_TIMING_PLOTS_CREATED = 0
_SCHEDULER_TIMING_SIGNATURES = set()

def _scheduler_timing_signature(timing):
    """Canonicalize a schedule for structural duplicate detection.

    Line order, whitespace, and a uniform shift of every start time are
    ignored. Operations, channels, relative timing, and durations remain
    significant, so genuinely different schedules still get plotted.
    """
    if isinstance(timing, str):
        lines = timing.splitlines()
    else:
        lines = [str(item) for item in timing]

    rows = []
    for line in lines:
        fields = tuple(field.strip() for field in line.split(","))
        if not any(fields):
            continue
        start = None
        if len(fields) > 3:
            try:
                start = float(fields[3])
            except ValueError:
                pass
        rows.append((fields, start))

    starts = [start for _, start in rows if start is not None]
    origin = min(starts) if starts else 0.0
    canonical = []
    for fields, start in rows:
        normalized = list(fields)
        if start is not None:
            normalized[3] = f"{start - origin:.12g}"
        canonical.append(tuple(normalized))
    return tuple(sorted(canonical))



def capture_scheduler_timing(result, label):
    """Display the first returned hardware schedule inside the notebook.

    IBM supplies scheduler timing only after a hardware PUB has completed.
    Status messages remain the live progress indicator while the job runs.
    """
    global _TIMING_PLOTS_CREATED
    if not SHOW_SCHEDULER_TIMING:
        return

    for pub_index, pub_result in enumerate(result):
        metadata = getattr(pub_result, "metadata", {}) or {}
        timing = (
            metadata.get("compilation", {})
            .get("scheduler_timing", {})
            .get("timing")
        )
        if not timing:
            continue
        signature = _scheduler_timing_signature(timing)
        if signature in _SCHEDULER_TIMING_SIGNATURES:
            continue
        if _TIMING_PLOTS_CREATED >= MAX_TIMING_PLOTS:
            return

        _SCHEDULER_TIMING_SIGNATURES.add(signature)
        _TIMING_PLOTS_CREATED += 1
        try:
            from qiskit_ibm_runtime.visualization import (
                draw_circuit_schedule_timing,
            )
            fig = draw_circuit_schedule_timing(
                circuit_schedule=timing,
                included_channels=None,
                filter_readout_channels=False,
                filter_barriers=False,
                width=1200,
            )
            fig.update_layout(title=f"Scheduler timing: {label}, PUB {pub_index}")
            try:
                fig.show(renderer="notebook")
            except ValueError as exc:
                # Plotly's notebook renderer requires nbformat.  Keep the
                # hardware workflow alive and display an HTML fallback.
                from IPython.display import HTML, display
                display(HTML(fig.to_html(
                    full_html=False, include_plotlyjs="cdn"
                )))
                print(
                    f"Used HTML timing fallback ({exc}). "
                    "Install nbformat>=4.2.0 for Plotly's notebook renderer.",
                    flush=True,
                )
        except (ImportError, ModuleNotFoundError) as exc:
            print(
                f"Scheduler timing visualization is unavailable: {exc}",
                flush=True,
            )
        return


def submit_sampler_job(sampler, circuits, shots, label):
    """Submit visibly, applying terminal-readout restrictions only to hardware."""
    backend = _sampler_runtime_backend(sampler)
    # Aer represents mid-circuit readout with ordinary measure instructions.
    # Gates after those measurements are valid in a dynamic simulation.
    if not _readout_is_simulator(backend):
        assert_terminal_measurements_are_final(circuits, label)
    print(
        f"[{label}] submitting {len(circuits)} circuit(s), "
        f"{int(shots)} shots each...",
        flush=True,
    )
    job = sampler.run(circuits, shots=int(shots))
    print(
        f"[{label}] accepted: job_id={_job_identifier(job)}",
        flush=True,
    )
    return job


def wait_for_sampler_job(job, label):
    """Poll quietly and print only when the job status changes."""
    terminal = {
        "DONE", "COMPLETED", "ERROR", "CANCELLED", "CANCELED",
        "STATUS UNAVAILABLE",
    }
    last_status = None
    while True:
        status = _job_status(job)
        if status != last_status:
            print(f"[{label}] status={status}", flush=True)
            last_status = status
        if status.upper() in terminal:
            break
        time.sleep(STATUS_POLL_SECONDS)
    result = job.result()
    print(f"[{label}] result received", flush=True)
    capture_scheduler_timing(result, label)
    return result



def _job_error_code_and_message(job, exc):
    """Return IBM's numeric error code and a compact failure message."""
    try:
        message = job.error_message() or str(exc)
    except Exception:
        message = str(exc)
    message = str(message).strip()
    match = re.search(r"Error code\s+(\d+)", message, flags=re.IGNORECASE)
    return (int(match.group(1)) if match else None), message



def _is_retryable_ibm_failure(code, message):
    """Retry bounded IBM availability, queue, service, and state failures."""
    text = str(message).lower()
    transient_markers = (
        "internal compiler error",
        "temporary internal error",
        "internal server error",
        "error queueing job",
        "queue is unavailable",
        "queue is disabled",
        "failed to run job",
        "out of sync",
        "unexpected state",
        "internal error syncing job",
        "job transition not valid",
        "job status not valid",
        "temporarily unavailable",
        "service unavailable",
        "error publishing job",
        "error creating job",
    )
    return (
        code in RETRYABLE_IBM_ERROR_CODES
        or any(marker in text for marker in transient_markers)
    )


def run_sampler_job_with_retries(sampler, circuits, shots, label):
    """Retry every caught submission/result error up to the attempt limit."""
    last_error = None
    retry_backend = _sampler_runtime_backend(sampler)
    for attempt in range(1, JOB_MAX_ATTEMPTS + 1):
        attempt_label = (
            label if JOB_MAX_ATTEMPTS == 1
            else f"{label}, attempt {attempt}/{JOB_MAX_ATTEMPTS}"
        )
        job = None
        try:
            # Keep preflight inside the retry boundary: status-query errors
            # must not bypass the all-errors retry policy.
            if attempt > 1 and retry_backend is not None:
                validate_runtime_backend(
                    retry_backend, wait_for_active=True
                )
            # Submission belongs inside the try block: queueing can fail before
            # Runtime returns a usable job object.
            job = submit_sampler_job(
                sampler, circuits, shots, attempt_label
            )
            return wait_for_sampler_job(job, attempt_label)
        except Exception as exc:
            last_error = exc
            code, message = _job_error_code_and_message(job, exc)
            if attempt >= JOB_MAX_ATTEMPTS:
                print(
                    f"[{label}] failed: job_id={_job_identifier(job)}, "
                    f"error_code={code}, retries_exhausted=True",
                    flush=True,
                )
                raise
            delay = JOB_RETRY_BASE_SECONDS * attempt
            print(
                f"[{label}] caught error {code}; retry_policy=all_errors; "
                f"job_id={_job_identifier(job)}. Retrying in {delay}s. "
                f"Server message: {message}",
                flush=True,
            )
            time.sleep(delay)
    raise last_error


def run_experiment_data_with_retries(
        submit_experiment, backend, label):
    """Retry a complete qiskit-experiments submission as a fresh workload."""
    last_error = None
    for attempt in range(1, JOB_MAX_ATTEMPTS + 1):
        attempt_label = f"{label}, attempt {attempt}/{JOB_MAX_ATTEMPTS}"
        active_job = None
        try:
            # Keep preflight inside the retry boundary so every ordinary
            # exception remains eligible for the next bounded attempt.
            if attempt > 1:
                validate_runtime_backend(backend, wait_for_active=True)
            print(
                f"[{attempt_label}] preparing and submitting experiment jobs...",
                flush=True,
            )
            experiment_data = submit_experiment()
            submitted_jobs = list(experiment_data.jobs())
            if submitted_jobs:
                print(
                    f"[{attempt_label}] accepted job(s): "
                    + ", ".join(
                        f"{_job_identifier(item)} ({_job_status(item)})"
                        for item in submitted_jobs
                    ),
                    flush=True,
                )
            for job_index, active_job in enumerate(submitted_jobs):
                wait_for_sampler_job(
                    active_job,
                    f"{attempt_label}, job {job_index}",
                )
            experiment_data.block_for_results()
            print(
                f"[{attempt_label}] experiment result received.",
                flush=True,
            )
            return experiment_data
        except Exception as exc:
            last_error = exc
            code, message = _job_error_code_and_message(active_job, exc)
            if attempt >= JOB_MAX_ATTEMPTS:
                print(
                    f"[{label}] experiment failed: "
                    f"job_id={_job_identifier(active_job)}, "
                    f"error_code={code}, retries_exhausted=True",
                    flush=True,
                )
                raise
            delay = JOB_RETRY_BASE_SECONDS * attempt
            print(
                f"[{label}] caught experiment error {code}; "
                "retry_policy=all_errors; "
                f"recreating the complete experiment in {delay}s. "
                f"Server message: {message}",
                flush=True,
            )
            time.sleep(delay)
    raise last_error


def append_hardware_error_history(backend, history_path):
    """Append one timestamped backend-calibration summary to a short text log."""
    from datetime import datetime
    from pathlib import Path
    from statistics import median

    def valid_probability(value):
        try:
            number = float(value)
        except (TypeError, ValueError):
            return None
        return number if np.isfinite(number) and 0.0 <= number <= 1.0 else None

    target = getattr(backend, "target", None)
    two_qubit_errors = []
    two_qubit_gate_names = set()
    seen_two_qubit_instructions = set()
    if target is not None:
        for operation_name in target.operation_names:
            try:
                instruction_entries = target[operation_name]
                entry_items = instruction_entries.items()
            except (KeyError, AttributeError, TypeError):
                continue
            for qargs, instruction_properties in entry_items:
                if qargs is None or len(qargs) != 2 or instruction_properties is None:
                    continue
                error = valid_probability(getattr(instruction_properties, "error", None))
                if error is None:
                    continue
                key = (str(operation_name), tuple(int(qubit) for qubit in qargs))
                if key in seen_two_qubit_instructions:
                    continue
                seen_two_qubit_instructions.add(key)
                two_qubit_errors.append(error)
                two_qubit_gate_names.add(str(operation_name))

    readout_errors = []
    if target is not None:
        try:
            measurement_items = target["measure"].items()
        except (KeyError, AttributeError, TypeError):
            measurement_items = ()
        for qargs, instruction_properties in measurement_items:
            if qargs is None or len(qargs) != 1 or instruction_properties is None:
                continue
            error = valid_probability(getattr(instruction_properties, "error", None))
            if error is not None:
                readout_errors.append(error)

    try:
        backend_properties = backend.properties()
    except Exception:
        backend_properties = None
    if not readout_errors and backend_properties is not None:
        for qubit in range(int(getattr(backend, "num_qubits", 0))):
            try:
                error = valid_probability(backend_properties.readout_error(qubit))
            except Exception:
                error = None
            if error is not None:
                readout_errors.append(error)

    backend_name = getattr(backend, "name", "unknown_backend")
    if callable(backend_name):
        backend_name = backend_name()
    queried_at = datetime.now().astimezone().isoformat(timespec="seconds")
    calibration_time = getattr(backend_properties, "last_update_date", None)
    if hasattr(calibration_time, "isoformat"):
        calibration_time = calibration_time.isoformat()
    calibration_time = str(calibration_time) if calibration_time is not None else "unavailable"
    median_qerror = median(two_qubit_errors) if two_qubit_errors else None
    median_readout = median(readout_errors) if readout_errors else None

    def format_value(value):
        return "unavailable" if value is None else f"{value:.10g}"

    history_path = Path(history_path)
    history_path.parent.mkdir(parents=True, exist_ok=True)
    lines = [
        f"[{queried_at}]",
        f"backend={backend_name}",
        f"calibration_last_update={calibration_time}",
        f"two_qubit_gate_names={','.join(sorted(two_qubit_gate_names)) or 'unavailable'}",
        f"median_two_qubit_qerror={format_value(median_qerror)}",
        f"two_qubit_qerror_samples={len(two_qubit_errors)}",
        f"median_readout_error={format_value(median_readout)}",
        f"readout_error_samples={len(readout_errors)}",
        "",
    ]
    with history_path.open("a", encoding="utf-8") as handle:
        handle.write("\n".join(lines) + "\n")
    print(f"Hardware calibration summary appended to {history_path}", flush=True)
    return {
        "backend": str(backend_name),
        "queried_at": queried_at,
        "calibration_last_update": calibration_time,
        "median_two_qubit_qerror": median_qerror,
        "median_readout_error": median_readout,
    }


def make_runtime_service():
    """Use a named IBM instance when configured to avoid ambiguous routing."""
    if RUNTIME_INSTANCE:
        print(f"Using IBM Quantum service instance: {RUNTIME_INSTANCE}")
        return QiskitRuntimeService(instance=RUNTIME_INSTANCE)
    print(
        "RUNTIME_INSTANCE is not set; IBM will choose among accessible "
        "service instances. Set it when your account has multiple projects."
    )
    return QiskitRuntimeService()


def validate_runtime_backend(
        backend, require_measure_2=False, wait_for_active=False):
    """Require IBM's authoritative ``operational + active`` state.

    The portal can display Online while the Runtime status endpoint still
    reports maintenance.  On a retry/resume path, wait and recheck every ten
    minutes instead of converting that transient state into a fatal error.
    Local simulators do not have a Runtime hardware status endpoint.
    """
    if _readout_is_simulator(backend):
        return backend
    while True:
        status = backend.status()
        operational = bool(getattr(status, "operational", False))
        pending_jobs = int(getattr(status, "pending_jobs", 0) or 0)
        status_message = str(
            getattr(status, "status_msg", "") or ""
        ).strip()
        active = operational and status_message.lower() == "active"
        print(
            f"Backend preflight: name={backend.name}, "
            f"operational={operational}, pending_jobs={pending_jobs}, "
            f"status={status_message or 'unknown'}",
            flush=True,
        )
        if active:
            break
        if not wait_for_active:
            raise RuntimeError(
                f"{backend.name} is not ready for a new workload "
                f"(operational={operational}, status={status_message!r}). "
                "Choose an active backend or wait for maintenance to finish."
            )
        print(
            f"{backend.name} is not active; checking again in "
            f"{MAINTENANCE_POLL_SECONDS // 60} minutes.",
            flush=True,
        )
        time.sleep(MAINTENANCE_POLL_SECONDS)
    if require_measure_2 and "measure_2" not in set(
            getattr(backend, "supported_instructions", ())):
        raise RuntimeError(
            f"{backend.name} does not advertise measure_2. Choose a backend "
            "that supports the MidCircuitMeasure instruction."
        )
    if pending_jobs >= BACKEND_PENDING_WARNING_THRESHOLD:
        print(
            f"WARNING: {backend.name} currently reports {pending_jobs} "
            "pending jobs. IBM error 1245 is more likely during heavy queue "
            "or service pressure; consider a less busy active backend.",
            flush=True,
        )
    return backend



def assert_terminal_measurements_are_final(circuits, label="workload"):
    """Reject active quantum work after an ordinary measurement.

    Runtime's dynamic scheduler can carry a measured qubit through a later
    control-flow container solely to align that container's branches. Such a
    formal qarg is not evidence that a gate acts on the measured qubit. Walk
    every branch and inspect its real instructions instead. Scheduler-only
    barriers and delays are harmless; gates, resets, control-flow work, and a
    second measurement on the same qubit are rejected.
    """
    circuit_items = [circuits] if hasattr(circuits, "data") else list(circuits)

    def walk_block(block, root_qubits, measured_at, pub_index, path):
        measured_at = dict(measured_at)
        for instruction_index, instruction in enumerate(block.data):
            operation = instruction.operation
            operation_name = operation.name
            nested_blocks = tuple(getattr(operation, "blocks", ()) or ())

            if nested_blocks:
                # A ControlFlowOp's formal qargs may include qubits that only
                # receive alignment delays. Map each block's local qubits to
                # the actual outer/root qubits and validate branch contents.
                outer_roots = [
                    root_qubits[block.find_bit(qubit).index]
                    for qubit in instruction.qubits
                ]
                branch_states = []
                for branch_index, nested_block in enumerate(nested_blocks):
                    if nested_block.num_qubits > len(outer_roots):
                        raise RuntimeError(
                            f"{label}: PUB {pub_index} has malformed "
                            f"{operation_name!r} qubit mapping at {path}."
                        )
                    branch_states.append(walk_block(
                        nested_block,
                        outer_roots[:nested_block.num_qubits],
                        measured_at,
                        pub_index,
                        f"{path}/{operation_name}[{branch_index}]",
                    ))

                # If an ordinary measurement occurs in any possible branch,
                # conservatively treat it as measured for subsequent work.
                for branch_state in branch_states:
                    for qubit, location in branch_state.items():
                        measured_at.setdefault(qubit, location)
                continue

            touched = {
                root_qubits[block.find_bit(qubit).index]
                for qubit in instruction.qubits
            }
            harmless_scheduler_ops = {"barrier", "delay"}
            violating = (
                touched.intersection(measured_at)
                if operation_name not in harmless_scheduler_ops
                else set()
            )
            if violating:
                details = ", ".join(
                    f"q{qubit} measured at {measured_at[qubit]}"
                    for qubit in sorted(violating)
                )
                raise RuntimeError(
                    f"{label}: PUB {pub_index} is unsafe: "
                    f"{operation_name!r} at {path}/instruction "
                    f"{instruction_index} acts after an ordinary terminal "
                    f"measurement ({details}). Submission stopped."
                )
            if operation_name == "measure":
                for qubit in touched:
                    measured_at[qubit] = (
                        f"{path}/instruction {instruction_index}"
                    )
        return measured_at

    for pub_index, item in enumerate(circuit_items):
        circuit = getattr(item, "circuit", None)
        if circuit is None and isinstance(item, (tuple, list)) and item:
            circuit = item[0]
        if circuit is None:
            circuit = item
        if not hasattr(circuit, "data"):
            raise TypeError(
                f"{label}: PUB {pub_index} does not contain a QuantumCircuit."
            )
        walk_block(
            circuit,
            list(range(circuit.num_qubits)),
            {},
            pub_index,
            "top-level",
        )
    return True


def _sampler_runtime_backend(sampler):
    """Return the IBM backend from a raw Sampler or notebook sampler proxy."""
    backend = getattr(sampler, "_backend", None)
    if backend is None:
        backend = getattr(getattr(sampler, "_sampler", None), "_backend", None)
    return backend
def _m3_calibration_sampler(backend, sampler):
    """Use a no-DD proxy for M3; preserve the experiment sampler and session.

    ManualDynamicDDSampler is defined later in the notebook. Plain hardware
    and simulator samplers used before that definition pass through unchanged.
    """
    dynamic_sampler_type = globals().get("ManualDynamicDDSampler")
    if (dynamic_sampler_type is None
            or not isinstance(sampler, dynamic_sampler_type)
            or sampler.policy == "none"):
        return sampler
    print(
        f"[M3 calibration] experiment policy={sampler.policy}; "
        "using policy=none for calibration; server built-in DD=False",
        flush=True,
    )
    # configure_mcm_policy_sampler already disables server DD. Reuse its raw
    # session sampler through a separate no-DD proxy, retaining ISA checks.
    # Do not change sampler.policy or its options: the sweep still needs DD.
    return dynamic_sampler_type(
        sampler._sampler, backend, "none", sampler.physical_qubits
    )


def calibrate_m3(
        backend, sampler, qubits, shots=M3_CALIBRATION_SHOTS,
        mid_circuit_qubits=(), label_suffix=None):
    """Build local M3 matrices without mixing measurement types in one job.

    IBM tunes ``measure_2`` separately from terminal ``measure``.  Each group
    is therefore calibrated in its own homogeneous Sampler job and the local
    assignment matrices are combined afterward.  This also avoids the mixed
    16-PUB payload that can fail at the IBM queue with error 1245.
    """
    # Prep-and-measure M3 circuits need no DD idle window. Route every MCM
    # calibration through no-DD, including legacy sweep entry points.
    sampler = _m3_calibration_sampler(backend, sampler)
    physical = list(dict.fromkeys(int(q) for q in qubits))
    mid_circuit_qubits = {int(q) for q in mid_circuit_qubits}
    unknown_mid = mid_circuit_qubits.difference(physical)
    if unknown_mid:
        raise ValueError(
            "measure_2 calibration qubits are outside the calibration set: "
            f"{sorted(unknown_mid)}"
        )

    calibration_groups = [("measure_2", physical, True)]

    matrices = [None] * int(backend.num_qubits)
    for calibration_kind, group_qubits, use_measure_2 in calibration_groups:
        width = len(group_qubits)
        order = 1
        while order < width + 1:
            order *= 2
        patterns = ((hadamard(order)[:, 1:width + 1] + 1) // 2).astype(int)
        shots_per_pattern = int(np.ceil(2 * int(shots) / len(patterns)))

        q = QuantumRegister(width, f"m3q_{calibration_kind}")
        c = ClassicalRegister(width, "m3cal")
        calibration_circuits = []
        for row_index, pattern in enumerate(patterns):
            qc = QuantumCircuit(q, c, name=f"m3_{calibration_kind}_{row_index:02d}")
            for logical, prepared_one in enumerate(pattern):
                if prepared_one:
                    qc.x(q[logical])
            for logical in range(width):
                if use_measure_2:
                    qc.append(MidCircuitMeasure(), [q[logical]], [c[logical]])
                else:
                    _append_measure_2(qc, q[logical], c[logical])
            calibration_circuits.append(qc)

        transpiled = transpile(
            calibration_circuits,
            backend=backend,
            initial_layout=group_qubits,
            optimization_level=0,
            seed_transpiler=SEED_TRANSPILER,
        )
        label = f"M3 {calibration_kind} calibration on {backend.name}"
        if label_suffix:
            label += f", {label_suffix}"
        result = run_sampler_job_with_retries(
            sampler, transpiled, shots_per_pattern, label
        )

        if len(result) != len(patterns):
            raise RuntimeError('M3 calibration result count does not match preparation patterns.')
        accumulators = [np.zeros((2, 2), dtype=float) for _ in group_qubits]
        for pub_result, pattern in zip(result, patterns):
            counts = pub_result.data.m3cal.get_counts()
            for key, count in counts.items():
                bits = normalize_bit_key(key, width)[::-1]
                for logical in range(width):
                    observed = int(bits[logical])
                    prepared = int(pattern[logical])
                    accumulators[logical][observed, prepared] += float(count)

        for logical, physical_qubit in enumerate(group_qubits):
            matrix = accumulators[logical]
            column_sums = matrix.sum(axis=0)
            if np.any(column_sums <= 0):
                raise RuntimeError(
                    f"Empty M3 calibration column for physical qubit "
                    f"{physical_qubit}: {matrix}."
                )
            matrices[physical_qubit] = matrix / column_sums
        print(
            f"[{label}] matrices ready for physical qubits {group_qubits}.",
            flush=True,
        )

    mitigator = mthree.M3Mitigation(backend)
    mitigator.cals_from_matrices(matrices)
    # Use M3's own calibration diagnostics and warning behavior.
    faulty = [int(qubit) for qubit in mitigator.faulty_qubits]
    fidelities = mitigator.readout_fidelity(physical)
    audit_path = _save_m3_calibration_audit({
        "backend": backend.name,
        "diagnostics_source": "mthree.M3Mitigation",
        "faulty_qubits": faulty,
        "calibrations": [
            {"physical_qubit": int(qubit),
             "matrix": np.asarray(matrix, dtype=float).tolist(),
             "readout_fidelity": float(fidelity)}
            for qubit, matrix, fidelity in zip(
                physical, [matrices[qubit] for qubit in physical], fidelities)
        ],
        "rows_are_observed_bits_columns_are_prepared_bits": True,
    }, OUTPUT_DIR)
    print(f"M3 built-in faulty qubits: {faulty}", flush=True)
    print(f"M3 readout fidelities: {dict(zip(physical, fidelities))}", flush=True)
    print(f"M3 diagnostic audit saved: {audit_path}", flush=True)
    print(
        f"M3 calibration complete for physical qubits {physical}.",
        flush=True,
    )
    return mitigator


def normalize_bit_key(key, width):
    if isinstance(key, (int, np.integer)):
        return format(int(key), f"0{width}b")
    text = str(key).replace(" ", "")
    if text.startswith(("0x", "0X")):
        text = format(int(text, 16), f"0{width}b")
    return text.zfill(width)


def probabilities_to_integer_counts(probabilities, shots):
    """Largest-remainder conversion required by tomography analysis."""
    keys = list(probabilities)
    probs = np.array(
        [max(0.0, float(probabilities[k])) for k in keys], dtype=float
    )
    if not keys or probs.sum() <= 0:
        raise ValueError("Empty corrected probability distribution.")
    probs /= probs.sum()
    scaled = probs * int(shots)
    counts = np.floor(scaled).astype(int)
    remainder = int(shots) - int(counts.sum())
    if remainder:
        order = np.argsort(-(scaled - counts))
        counts[order[:remainder]] += 1
    return {k: int(v) for k, v in zip(keys, counts) if v}


def _apply_m3_quasi(counts, measured_physical, mitigator):
    """Apply M3 with classical-bit 0 mapped to the rightmost count bit."""
    width = len(measured_physical)
    cleaned = {
        normalize_bit_key(key, width): int(value)
        for key, value in counts.items()
    }
    lengths = {len(key) for key in cleaned}
    if lengths != {width}:
        raise ValueError(
            f"Expected {width}-bit counts, received lengths {lengths}."
        )
    return mitigator.apply_correction(cleaned, list(measured_physical), distance=-1)





def m3_probability_distribution(counts, measured_physical, mitigator):
    """Return a physical M3 distribution for tomography reconstruction."""
    width = len(measured_physical)
    quasi = _apply_m3_quasi(counts, measured_physical, mitigator)
    nearest = quasi.nearest_probability_distribution()
    out = {}
    for key, value in nearest.items():
        bitstring = normalize_bit_key(key, width)
        out[bitstring] = out.get(bitstring, 0.0) + float(value)
    return out



def sampler_counts_for_register(result, register_name):
    counts = []
    for pub_result in result:
        register_data = getattr(pub_result.data, register_name)
        counts.append(register_data.get_counts())
    return counts


def sampler_joint_dynamic_counts(result, register_names):
    """Join Sampler registers into Qiskit's global classical-bit order.

    ``BitArray.concatenate_bits([low_register, ..., high_register])`` emits
    the high register on the left, exactly like a Qiskit count string.  The
    circuit's register-creation order must therefore be passed directly and
    must not be reversed.  Reversing it maps M3 calibrations to the wrong bits
    and corrupts the reference/receiver outcome mapping.
    """
    names = list(register_names)
    counts = []
    for pub_result in result:
        data = pub_result.data
        joined = BitArray.concatenate_bits([
            getattr(data, name) for name in names
        ])
        counts.append(joined.get_counts())
    return counts


def _validate_sampler_register_order_contract():
    """Fail early if the installed BitArray ordering contract changes."""
    low = BitArray.from_samples(["01"], num_bits=2)
    middle = BitArray.from_samples(["10"], num_bits=2)
    high = BitArray.from_samples(["11"], num_bits=2)
    observed = BitArray.concatenate_bits([low, middle, high]).get_bitstrings()[0]
    expected = "111001"  # high + middle + low
    if observed != expected:
        raise RuntimeError(
            "Unexpected BitArray.concatenate_bits ordering: "
            f"expected {expected}, observed {observed}."
        )


_validate_sampler_register_order_contract()




print("Backend and mitigation helpers loaded.")

## Dynamic scheduling and XX/XY4 idle-time protection

In [ ]:
_MCM_DD_SCHEDULE_AUDIT = []

def _recursive_count_ops(circuit):
    try:
        return dict(circuit.count_ops(recurse=True))
    except TypeError:
        return dict(circuit.count_ops())


def _backend_pulse_alignment(backend, target):
    """Resolve pulse alignment from the IBM backend, then the target."""
    try:
        constraints = backend.configuration().timing_constraints or {}
    except Exception:
        constraints = {}
    return int(
        constraints.get("pulse_alignment")
        or getattr(target, "pulse_alignment", None)
        or 16
    )


def build_mcm_scheduling_pipeline(backend, policy, physical_qubits):
    """Build the safe local scheduling stage used by MCM DD policies.

    The complete dynamic circuit is first scheduled with ALAP and PadDelay,
    with end-of-circuit padding disabled.  DD gates are inserted afterward by
    ``_insert_control_flow_safe_dd`` only into top-level delays that precede
    every ordinary terminal measurement.  This avoids PadDynamicalDecoupling's
    block-end padding, which previously placed XY4 gates after tomography
    measurements and caused IBM 9699 failures.
    """
    if policy not in DD_POLICIES:
        raise ValueError(f"Unknown DD policy {policy!r}; choose {DD_POLICIES}.")

    target = copy.deepcopy(backend.target)
    native_basis = list(dict.fromkeys([
        *backend.operation_names,
        "if_else", "while_loop", "for_loop", "switch_case",
    ]))
    pulse_alignment = _backend_pulse_alignment(backend, target)
    translate_pm = None

    # Y is used only as a temporary logical marker.  Scheduling is based on
    # the calibrated X duration, and the marker is translated after insertion
    # to X followed by the zero-duration virtual frame change RZ(pi).
    if policy == "XY4" and "y" not in target.operation_names:
        if "x" not in target.operation_names:
            raise RuntimeError("Backend target has no calibrated X gate.")
        xy4_equivalences = EquivalenceLibrary(base=sel)
        y_native = QuantumCircuit(1, name="native_y_for_xy4")
        y_native.x(0)
        y_native.rz(np.pi, 0)
        xy4_equivalences.add_equivalence(YGate(), y_native)
        translate_pm = PassManager([
            BasisTranslator(xy4_equivalences, native_basis)
        ])

    schedule_pm = PassManager([
        ALAPScheduleAnalysis(target=target),
        PadDelay(
            target=target,
            fill_very_end=False,
            schedule_idle_qubits=False,
        ),
    ])
    return schedule_pm, translate_pm, pulse_alignment


def _assert_dynamic_structure_preserved(before, after, policy, pub_index):
    """Fail if scheduling alters measure_2 or feedback control flow."""
    for operation in ("measure_2", "if_else"):
        if before.get(operation, 0) != after.get(operation, 0):
            raise RuntimeError(
                f"MCM {policy} PUB {pub_index}: scheduling changed "
                f"{operation} count from {before.get(operation, 0)} to "
                f"{after.get(operation, 0)}. Submission stopped."
            )


def _append_mcm_schedule_audit(rows):
    _MCM_DD_SCHEDULE_AUDIT.extend(rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(_MCM_DD_SCHEDULE_AUDIT).to_csv(
        MCM_DD_AUDIT_CSV, index=False
    )


def _feedback_target_qubits(circuits):
    """Physical qubits directly addressed by control-flow bodies."""
    protected = set()
    for circuit in circuits:
        for item in circuit.data:
            if isinstance(item.operation, ControlFlowOp):
                protected.update(
                    circuit.find_bit(qubit).index
                    for qubit in item.qubits
                )
    return protected


def _nested_operation_count(circuit, operation_name):
    """Count an operation only inside control-flow block bodies."""
    total = 0

    def count_block(block):
        nonlocal total
        for item in block.data:
            if item.operation.name == operation_name:
                total += 1
            if isinstance(item.operation, ControlFlowOp):
                for nested in item.operation.blocks:
                    count_block(nested)

    for item in circuit.data:
        if isinstance(item.operation, ControlFlowOp):
            for block in item.operation.blocks:
                count_block(block)
    return total


def _assert_global_terminal_measurement_boundary(circuits, label):
    """Reject active work anywhere after an ordinary terminal measurement.

    IBM ``measure`` is reserved for the final acquisition block in these MCM
    workloads.  Mid-circuit acquisition is represented by ``measure_2``.
    Barriers and delays are harmless, but no gate, reset, or control-flow
    operation may follow the first ordinary measure in a block.
    """
    circuit_items = [circuits] if hasattr(circuits, "data") else list(circuits)

    def check_block(block, pub_index, path):
        terminal_started = False
        for instruction_index, item in enumerate(block.data):
            operation = item.operation
            name = operation.name
            if terminal_started and name not in {"measure", "barrier", "delay"}:
                raise RuntimeError(
                    f"{label}: PUB {pub_index} has active instruction {name!r} "
                    f"after the ordinary terminal-measurement block began at "
                    f"{path}/instruction {instruction_index}. Submission stopped."
                )
            if name == "measure":
                terminal_started = True
            for branch_index, nested in enumerate(
                    tuple(getattr(operation, "blocks", ()) or ())):
                check_block(
                    nested, pub_index,
                    f"{path}/{name}[{branch_index}]",
                )

    for pub_index, item in enumerate(circuit_items):
        circuit = getattr(item, "circuit", None)
        if circuit is None and isinstance(item, (tuple, list)) and item:
            circuit = item[0]
        if circuit is None:
            circuit = item
        check_block(circuit, pub_index, "top-level")
    return True


def _insert_control_flow_safe_dd(
        circuit, backend, allowed_qubits, pulse_alignment, policy):
    """Replace eligible top-level delays with an XX or XY4 identity cycle.

    Control-flow bodies are copied without modification.  DD is never inserted
    after ``measure_2`` on the measured wire, after an ordinary terminal
    measurement anywhere, or into end-of-circuit padding.  Every replacement
    has exactly the same duration as the delay it replaces.
    """
    if policy == "XX":
        sequence = ("x", "x")
        spacing = np.asarray([0.25, 0.50, 0.25], dtype=float)
    elif policy == "XY4":
        sequence = ("x", "y", "x", "y")
        spacing = np.asarray([0.125, 0.25, 0.25, 0.25, 0.125], dtype=float)
    else:
        raise ValueError(f"Manual DD insertion is unavailable for {policy!r}.")

    allowed = set(map(int, allowed_qubits))
    alignment = max(1, int(pulse_alignment))
    durations = backend.target.durations()
    rewritten = circuit.copy_empty_like()
    measured_qubits = set()
    terminal_started = False
    cycles = 0
    boundary_cycles = {}
    skipped_after_measure_2 = 0
    skipped_after_terminal_measure = 0

    for item in circuit.data:
        qindices = [circuit.find_bit(q).index for q in item.qubits]
        cindices = [circuit.find_bit(c).index for c in item.clbits]
        new_qargs = [rewritten.qubits[index] for index in qindices]
        new_cargs = [rewritten.clbits[index] for index in cindices]
        operation = item.operation
        name = operation.name

        if name == "measure":
            terminal_started = True

        eligible_delay = (
            name == "delay"
            and len(qindices) == 1
            and qindices[0] in allowed
            and operation.unit == "dt"
        )
        if eligible_delay and terminal_started:
            skipped_after_terminal_measure += 1
        if eligible_delay and qindices[0] in measured_qubits:
            skipped_after_measure_2 += 1

        if (
            not eligible_delay
            or terminal_started
            or qindices[0] in measured_qubits
        ):
            rewritten.append(operation, new_qargs, new_cargs)
        else:
            qubit = qindices[0]
            interval = int(operation.duration)
            try:
                x_duration = int(round(
                    durations.get("x", qubit, unit="dt")
                ))
            except Exception:
                rewritten.append(operation, new_qargs, new_cargs)
                x_duration = 0

            sequence_duration = len(sequence) * x_duration
            if (
                x_duration <= 0
                or x_duration % alignment != 0
                or interval / sequence_duration <= 2.0
            ):
                if x_duration > 0:
                    rewritten.append(operation, new_qargs, new_cargs)
            else:
                slack = interval - sequence_duration
                taus = (
                    alignment * np.floor(slack * spacing / alignment)
                ).astype(int)
                extra = int(slack - int(taus.sum()))
                middle = (len(taus) - 1) // 2
                to_middle = alignment * (extra // alignment)
                taus[middle] += to_middle
                taus[-1] += extra - to_middle
                if int(taus.sum()) + sequence_duration != interval:
                    raise RuntimeError(
                        f"Control-flow-safe {policy} changed delay "
                        f"duration {interval}."
                    )

                for gate_index, gate_name in enumerate(sequence):
                    if taus[gate_index] > 0:
                        rewritten.delay(
                            int(taus[gate_index]), new_qargs[0], unit="dt"
                        )
                    if gate_name == "x":
                        rewritten.x(new_qargs[0])
                    else:
                        rewritten.y(new_qargs[0])
                if taus[-1] > 0:
                    rewritten.delay(int(taus[-1]), new_qargs[0], unit="dt")
                cycles += 1

        if name in {"measure", "measure_2"}:
            measured_qubits.update(qindices)
        elif name == "reset":
            measured_qubits.difference_update(qindices)
        label = getattr(operation, 'label', None)
        if name == 'barrier' and label in {f'CUMULATIVE_ROUND_{r}' for r in range(MAX_ROUNDS + 1)}:
            boundary_cycles[int(label.rsplit('_', 1)[1])] = cycles

    # XYXY = -I.  Restore exact circuit equivalence, including global phase.
    if policy == "XY4" and cycles % 2:
        rewritten.global_phase += np.pi

    return rewritten, {
        "dd_cycles": cycles,
        "dd_cycles_by_round": boundary_cycles,
        "skipped_delays_after_measure_2": skipped_after_measure_2,
        "skipped_delays_after_terminal_measure": skipped_after_terminal_measure,
    }

## State serialization and reference–receiver identities

In [ ]:
from qiskit.quantum_info import DensityMatrix, state_fidelity

def _sanitize_two_qubit_density_matrix(rho):
    """Hermitize, normalize, and project tiny tomography negativity to PSD."""
    rho = np.asarray(rho, dtype=complex).reshape(4, 4)
    hermitian = 0.5 * (rho + rho.conj().T)
    trace = float(np.real(np.trace(hermitian)))
    if not np.isfinite(trace) or abs(trace) < 1e-12:
        raise ValueError("Tomography returned a zero/non-finite trace.")
    hermitian /= trace
    eigenvalues, eigenvectors = np.linalg.eigh(hermitian)
    clipped = np.clip(np.real(eigenvalues), 0.0, None)
    if clipped.sum() <= 0:
        raise ValueError("Tomography density matrix has no positive spectrum.")
    projected = (eigenvectors * clipped) @ eigenvectors.conj().T
    projected /= np.trace(projected)
    projection = float(np.linalg.norm(projected - hermitian, ord="fro"))
    return projected, projection, float(np.min(eigenvalues))


def _rho_payload(rho):
    physical, projection, minimum_eigenvalue = _sanitize_two_qubit_density_matrix(rho)
    return {
        "rho_real": physical.real.tolist(),
        "rho_imag": physical.imag.tolist(),
        "rho_psd_projection_frobenius": projection,
        "rho_preprojection_min_eigenvalue": minimum_eigenvalue,
    }


def _rho_from_record(record):
    physical, _, _ = _sanitize_two_qubit_density_matrix(
        np.asarray(record["rho_real"], dtype=float)
        + 1j * np.asarray(record["rho_imag"], dtype=float)
    )
    return physical


def _input_rho_from_record(record):
    physical, _, _ = _sanitize_two_qubit_density_matrix(
        np.asarray(record["input_rho_real"], dtype=float)
        + 1j * np.asarray(record["input_rho_imag"], dtype=float)
    )
    return physical
def trajectory_state_pair_spec(full_layout, channel, round_number):
    """Return an auditable identity for the state measured by both analyses."""
    channel = str(channel)
    round_number = int(round_number)
    if channel not in TRAJECTORY_STATE_PAIRS:
        raise ValueError(f"Unknown channel {channel!r}.")
    left, right, full_indices, label = TRAJECTORY_STATE_PAIRS[channel][round_number]
    physical = [int(full_layout[index]) for index in full_indices]
    analysis_indices = [0, 1] if round_number == 0 else list(full_indices)
    return {
        "state_pair_label": label,
        "state_qubit_left_role": left,
        "state_qubit_right_role": right,
        "full_circuit_logical_indices": list(full_indices),
        "analysis_circuit_logical_indices": analysis_indices,
        "state_physical_qubits": physical,
    }


def _state_pair_record_fields(state_pair):
    if state_pair is None:
        raise ValueError("Every fidelity record requires state-pair metadata.")
    return {
        "state_pair_label": state_pair["state_pair_label"],
        "state_qubit_left_role": state_pair["state_qubit_left_role"],
        "state_qubit_right_role": state_pair["state_qubit_right_role"],
        "state_full_logical_indices": json.dumps(state_pair["full_circuit_logical_indices"]),
        "state_analysis_logical_indices": json.dumps(state_pair["analysis_circuit_logical_indices"]),
        "state_physical_qubits": json.dumps(state_pair["state_physical_qubits"]),
        "tomography_reference_receiver_pair": True,
    }

## Classical Pauli frames and overlap-enabled five-link resource construction


In [ ]:
"""Notebook helpers for physical feed-forward and offline Pauli frames."""

CORRECTION_METHODS = ('classical',)
CORRECTION_METHOD = 'classical'
CORRECTION_LABELS = {'classical': 'Classical Pauli post-processing'}




def _pauli_frame_pair_distribution(distribution, width, channel, round_number, receiver_basis):
    """Correct each joint outcome, then marginalize to reference/receiver.

    Global classical-bit order (LSB first):
      r1_uA, r1_vA, r1_uB, r1_vB, ..., analysis_ref, analysis_receiver.
    Teleporting through identity channels accumulates a frame X^x Z^z,
    x = XOR_r v_r, z = XOR_r u_r, independently for each channel.
    PauliMeasurementBasis indices are 0=Z, 1=X, 2=Y. Flip only the
    receiver's tomography result when its measured Pauli anticommutes
    with the accumulated frame. Keep ALL branches and ALL shots.
    """
    round_number = int(round_number)
    if channel not in ("A", "B") or width != 4 * round_number + 2:
        raise ValueError("Unexpected channel or joint classical-register width.")
    if receiver_basis not in (0, 1, 2):
        raise ValueError("Expected the standard Z/X/Y tomography basis indices.")
    channel_offset = 0 if channel == "A" else 2
    pair = {}
    for key, value in distribution.items():
        text = normalize_bit_key(key, width)
        if len(text) != width or set(text) - {"0", "1"}:
            raise ValueError("Malformed joint outcome key.")
        bits = list(map(int, text[::-1]))
        x_frame = z_frame = 0
        for r in range(round_number):
            offset = 4 * r + channel_offset
            z_frame ^= bits[offset]
            x_frame ^= bits[offset + 1]
        flip = (x_frame, z_frame, x_frame ^ z_frame)[int(receiver_basis)]
        receiver = bits[-1] ^ flip
        reference = bits[-2]
        pair_key = f"{receiver}{reference}"  # Qiskit counts: c1 then c0.
        pair[pair_key] = pair.get(pair_key, 0.0) + float(value)
    total = sum(pair.values())
    if not np.isfinite(total) or total <= 0:
        raise ValueError("Empty or invalid frame-corrected distribution.")
    return {key: value / total for key, value in pair.items()}


def _m3_recover_joint_distributions(counts, mapping, state, label):
    """Mitigate ALL Bell bits and analysis bits before frame correction.

    Unlike unconditional feed-forward tomography, dropping a faulty Bell bit
    here would destroy information needed for the Pauli frame. Recovery may
    recalibrate the complete required set, but must never silently discard
    syndrome bits or substitute raw counts for an M3 result.
    """
    mapping = list(map(int, mapping))
    cache_key = ("classical_joint", tuple(mapping))

    def correct(mitigator):
        bad = _m3_unusable_qubits(mitigator, mapping)
        if bad:
            raise ValueError(f"Unusable full-joint M3 calibration: {bad}")
        distributions = [m3_probability_distribution(c, mapping, mitigator) for c in counts]
        normalized = []
        for distribution in distributions:
            values = np.asarray(list(distribution.values()), dtype=float)
            if (not values.size or not np.isfinite(values).all()
                    or np.any(values < -1e-10) or values.sum() <= 0):
                raise ValueError("Invalid full-joint M3 probability distribution.")
            total = float(values.sum())
            normalized.append({key: float(value / total) for key, value in distribution.items()})
        return normalized

    initial = state["recovered"].get(cache_key, state["original"])
    try:
        return correct(initial)
    except Exception as exc:
        state["recovered"].pop(cache_key, None)
        _m3_recovery_note(state, label, attempt=0, action="full_joint_correction_failed",
                          error=str(exc), required_physical=mapping)
    for attempt, multiplier in enumerate((1, 2, 4), 1):
        try:
            mitigator = calibrate_m3(
                state["backend"], state["sampler"], sorted(set(mapping)),
                shots=int(M3_CALIBRATION_SHOTS) * multiplier,
                label_suffix=f"classical full-joint recovery {attempt}: {label}")
            result = correct(mitigator)
            state["recovered"][cache_key] = mitigator
            _m3_recovery_note(state, label, attempt=attempt,
                              action="full_joint_recovery_succeeded",
                              required_physical=mapping, excluded_classical_bits=[])
            return result
        except Exception as exc:
            _m3_recovery_note(state, label, attempt=attempt,
                              action="full_joint_recovery_failed", error=str(exc))
    raise RuntimeError("Full-joint M3 failed; keep raw data and omit this M3 result. "
                       "Bell bits cannot be discarded for classical correction.")


def _tomography_pair_distributions(counts, mapping, state, suffixes, channel,
                                   round_number, label, method):
    """One nine-setting acquisition supplies both raw and M3 reconstructions."""
    if len(counts) != 9 or len(suffixes) != 9:
        raise ValueError("Exactly nine two-qubit tomography settings are required.")
    width = len(mapping)
    if CORRECTION_METHOD != "classical":
        raise ValueError("Unknown Pauli-correction method.")
    joint = counts if method == "raw" else _m3_recover_joint_distributions(counts, mapping, state, label)
    pairs = []
    for distribution, suffix in zip(joint, suffixes):
        indices = list(suffix.metadata.get("m_idx", ()))
        if len(indices) != 2:
            raise ValueError("Missing reference/receiver tomography basis metadata.")
        pairs.append(_pauli_frame_pair_distribution(
            distribution, width, channel, round_number, int(indices[1])))
    return pairs


def append_mcm_cascade_record(records, trial, method, channel, round_number, rho,
                               policy, domain="Hardware", input_rho=None,
                               input_state_source="", state_pair=None,
                               input_state_available=True):
    physical, _, _ = _sanitize_two_qubit_density_matrix(rho)
    record = {
        "measurement_variant": EXPERIMENT_VARIANT,
        "correction_method": CORRECTION_METHOD,
        "correction_label": CORRECTION_LABELS[CORRECTION_METHOD],
        "pauli_corrections_on_device": False,
        "pauli_frame_rounds": int(round_number) if CORRECTION_METHOD == "classical" else 0,
        "tomography_state_frame": "classically_corrected",
        "next_round_input_carries_pauli_frame": CORRECTION_METHOD == "classical" and 0 < round_number < MAX_ROUNDS,
        "bell_bits_required_for_m3": CORRECTION_METHOD == "classical" and round_number > 0,
        "trial": int(trial), "domain": domain, "method": method,
        "measurement_mitigation_engine": MTHREE_IMPLEMENTATION if method == "m3" else "none",
        "series_key": f"{CORRECTION_METHOD}_{policy}_{method}",
        "dd_policy": policy, "channel": channel, "round": int(round_number),
        "shots_per_tomography_basis": int(SHOTS), "tomography_settings": 9,
        "state_role": _state_role(round_number),
        "round_1_output_is_round_2_input": True,
        "round_2_output_is_round_3_input": True,
        "fidelity": metrics_from_density_matrix(physical),
    }
    record.update(_state_pair_record_fields(state_pair))
    record.update(_rho_payload(physical))
    record["input_state_available"] = bool(input_state_available)
    if input_state_available:
        input_payload = _rho_payload(physical if input_rho is None else input_rho)
        for key, value in input_payload.items():
            record["input_" + key] = value
    else:
        record.update(input_rho_real=np.nan, input_rho_imag=np.nan,
                      input_rho_psd_projection_frobenius=np.nan,
                      input_rho_preprojection_min_eigenvalue=np.nan)
    record["input_state_source"] = input_state_source
    records.append(record)

"""Standalone notebook helpers: one overlap-enabled classical implementation.

These definitions are embedded in the notebooks, not imported at run time.
No helper connects to hardware or submits jobs.
"""

_CLASSICAL_PARALLEL_VERSION = 'classical_parallel_round_count_v3_R45'


def build_parallel_classical_cascade(rounds=MAX_ROUNDS):
    """Keep the I -> o1 -> ... -> o5 wiring, without intermediate corrections.

    All resources needed at this checkpoint are prepared/decoded first.
    Independent Bell CNOTs, then Hadamards, then measure_2 are grouped into
    layers. There are no per-round barriers. Round numbers denote the
    number of consumed links, NOT strict scheduled prefixes of one master.
    One 29-logical-qubit initial assignment is retained at every checkpoint.
    """
    if rounds not in range(MAX_ROUNDS + 1):
        raise ValueError("Round outside 0..MAX_ROUNDS.")
    refs = [QuantumRegister(1, f"ref_{channel}") for channel in ("A", "B")]
    inputs = [QuantumRegister(1, f"in_{channel}") for channel in ("A", "B")]
    resources = [QuantumRegister(5, f"r{r}") for r in range(1, MAX_ROUNDS + 1)]
    syndromes = [(ClassicalRegister(2, f"r{r}_mA"), ClassicalRegister(2, f"r{r}_mB"))
                 for r in range(1, rounds + 1)]
    anchor = ClassicalRegister(4, "shared_anchor")
    circuit = QuantumCircuit(*refs, *inputs, *resources,
                             *(reg for pair in syndromes for reg in pair), anchor,
                             name=f"rank{DISENTANGLER_RANK}_classical_parallel_R{rounds}")
    for ref, inp in zip(refs, inputs):
        circuit.h(ref[0])
        circuit.cx(ref[0], inp[0])
    for resource in resources[:rounds]:
        _apply_ame52_prep(circuit, resource)
        apply_V234_dagger(circuit, resource)
    current = [reg[0] for reg in inputs]
    bell_measurements = []
    for resource, bits_by_channel in zip(resources[:rounds], syndromes):
        for index, (bits, (_, sender_index, receiver_index)) in enumerate(
                zip(bits_by_channel, RANK_CHANNEL_RESOURCE_ROLES)):
            inp, sender = current[index], resource[sender_index]
            bell_measurements.append((inp, sender, bits[0], bits[1]))
            current[index] = resource[receiver_index]
    participants = [q for inp, sender, _, _ in bell_measurements for q in (inp, sender)]
    if len(set(participants)) != 4 * rounds:
        raise RuntimeError("Bell measurements across channels/rounds are not disjoint.")
    for inp, sender, _, _ in bell_measurements:
        circuit.cx(inp, sender)
    for inp, _, _, _ in bell_measurements:
        circuit.h(inp)
    if rounds:
        # A single common readout layer prevents routing from consuming an
        # early link while still doing SWAPs/gates for later Bell measurements.
        # This is NOT a round boundary or another scheduling experiment.
        circuit.barrier(*circuit.qubits, label="CLASSICAL_PARALLEL_BELL_READOUT_LAYER")
    for inp, sender, bit_u, bit_v in bell_measurements:
        _append_measure_2(circuit, inp, bit_u)
        _append_measure_2(circuit, sender, bit_v)
    # The final boundary is needed to attach tomography and to end DD.
    # These anchor measurements protect/identify the routed state locations;
    # they are stripped before the nine existing analyzers are attached.
    circuit.barrier(*circuit.qubits, label=f"CUMULATIVE_ROUND_{rounds}")
    pairs = {channel: [circuit.find_bit(refs[index][0]).index,
                       circuit.find_bit(current[index]).index]
             for index, channel in enumerate(("A", "B"))}
    _append_measure_2(circuit,
                     [refs[0][0], current[0], refs[1][0], current[1]], list(anchor))
    circuit.metadata = {
        "teleportation_rounds": rounds, "disentangler_rank": DISENTANGLER_RANK,
        "correction_method": "classical", "pauli_corrections_on_device": False,
        "preparation_schedule": _CLASSICAL_PARALLEL_VERSION,
        "physical_prefixes_verified": False,
        "checkpoint_construction": "independently_scheduled_logical_round_count",
        "round_1_output_is_round_2_input": rounds >= 2,
        "round_2_output_is_round_3_input": rounds >= 3,
        "previous_output_is_next_input": True,
        "receiver_input_chain": [list(range(1, rounds + 1))] * 2,
        "logical_reference_receiver_pairs": pairs,
        "bell_measurement_logical_pairs": [
            [circuit.find_bit(inp).index, circuit.find_bit(sender).index]
            for inp, sender, _, _ in bell_measurements],
    }
    return circuit


def _classical_parallel_anchor_pairs(circuit, round_number):
    """Read actual physical state locations from routed anchor instructions."""
    offset = 4 * int(round_number)
    found = {}
    for item in circuit.data:
        if item.operation.name in ("measure", "measure_2"):
            bit = circuit.find_bit(item.clbits[0]).index
            if offset <= bit < offset + 4:
                if bit in found:
                    raise RuntimeError("Duplicate classical-parallel anchor measurement.")
                found[bit] = circuit.find_bit(item.qubits[0]).index
    if set(found) != set(range(offset, offset + 4)) or len(set(found.values())) != 4:
        raise RuntimeError("Missing or overlapping classical-parallel state anchors.")
    return {"A": [found[offset], found[offset + 1]],
            "B": [found[offset + 2], found[offset + 3]]}


def _classical_parallel_compile_family(backend, full_layout, strategy):
    """Compile R4/R5 locally; same experiment/job grid, a single ALAP policy.

    The classical path has its own routed circuits, cached across levels.
    All endpoints use the same initial 29-qubit layout. Final routed maps
    may differ and are reported honestly, including all Bell-bit locations.
    """
    if len(full_layout) != N_LOGICAL_QUBITS or len(set(full_layout)) != N_LOGICAL_QUBITS:
        raise ValueError("Classical parallel compilation needs 29 distinct physical qubits.")
    if set(DD_POLICIES) != {"none", "XX", "XY4"}:
        raise ValueError("Expected the existing none/XX/XY4 experiment grid.")
    strategy = dict(strategy, scheduling_method="alap")
    family = {
        "version": _CLASSICAL_PARALLEL_VERSION, "prefixes": {}, "stats": {},
        "masters": {}, "mappings": {}, "pairs": {}, "ancestors": {},
        "recovery_strategy": strategy, "failures": [], "correction_method": "classical",
        "initial_physical_layout": list(full_layout),
        "physical_prefixes_verified": False,
        "checkpoint_construction": "independently_scheduled_logical_round_count",
        "preparation_schedule": _CLASSICAL_PARALLEL_VERSION,
        "routing_sha256_by_round": {}, "timing_by_round": {},
    }
    failed_policies = set()
    for r in CHECKPOINT_ROUNDS:
        template = build_parallel_classical_cascade(r)
        route_key = (_CLASSICAL_PARALLEL_VERSION, id(backend), backend.name,
                     strategy["seed_transpiler"], tuple(full_layout), r,
                     _shared_digest(template))
        if route_key not in _SHARED_ROUTED_CACHE:
            routed = transpile(template, backend=backend, initial_layout=list(full_layout),
                               optimization_level=0, seed_transpiler=strategy["seed_transpiler"],
                               approximation_degree=1.0, routing_method="sabre")
            expanded = routed.copy_empty_like()
            for item in routed.data:
                if (item.operation.name == "barrier"
                        and getattr(item.operation, "label", None) in {
                            f"CUMULATIVE_ROUND_{r}", "CLASSICAL_PARALLEL_BELL_READOUT_LAYER"}):
                    expanded.barrier(*expanded.qubits, label=item.operation.label)
                else:
                    expanded.append(item.operation, item.qubits, item.clbits)
            expanded._layout = routed.layout
            expanded.metadata = copy.deepcopy(routed.metadata or {})
            _SHARED_ROUTED_CACHE[route_key] = expanded
        routed = _SHARED_ROUTED_CACHE[route_key]
        pairs = _classical_parallel_anchor_pairs(routed, r)
        syndrome_map = _shared_syndrome_map(routed, 4 * r)
        if (len(set(syndrome_map)) != 4 * r
                or set(syndrome_map) & {q for pair in pairs.values() for q in pair}):
            raise RuntimeError("Routed simultaneous Bell readouts are not disjoint from the final receivers.")
        digest = _shared_digest(routed)
        family["routing_sha256_by_round"][r] = digest
        isa = transpile(routed, backend=backend,
                        initial_layout=list(range(routed.num_qubits)), routing_method="none",
                        optimization_level=OPTIMIZATION_LEVEL,
                        seed_transpiler=strategy["seed_transpiler"], approximation_degree=1.0)
        if (_classical_parallel_anchor_pairs(isa, r) != pairs
                or _shared_syndrome_map(isa, 4 * r) != syndrome_map):
            raise RuntimeError("Native optimization changed the classical-parallel readout map.")
        schedule, translator, alignment = build_mcm_scheduling_pipeline(backend, "XY4", full_layout)
        master = schedule.run(isa)
        if (_classical_parallel_anchor_pairs(master, r) != pairs
                or _shared_syndrome_map(master, 4 * r) != syndrome_map
                or _shared_counts(master).get("if_else", 0)):
            raise RuntimeError("Scheduling changed classical-parallel state/measurement locations.")
        family["pairs"][r] = pairs
        family["mappings"][r] = {channel: syndrome_map + pairs[channel]
                                 for channel in ("A", "B")}
        active = _shared_active_qubits(master)
        family["timing_by_round"][r] = _classical_parallel_timing(master, backend, r)
        for policy in ["none"] + [p for p in DD_POLICIES if p != "none"]:
            if policy in failed_policies:
                continue
            try:
                if policy == "none":
                    prepared, cycles = master.copy(), 0
                else:
                    prepared, stats = _insert_control_flow_safe_dd(
                        master, backend, active, alignment, policy)
                    cycles = stats["dd_cycles_by_round"][r]
                    if policy == "XY4" and translator is not None:
                        prepared = translator.run(prepared)
                if (_classical_parallel_anchor_pairs(prepared, r) != pairs
                        or _shared_syndrome_map(prepared, 4 * r) != syndrome_map
                        or _shared_counts(prepared).get("if_else", 0)
                        or _shared_counts(prepared).get("measure_2", 0) != 4 * r + 4):
                    raise RuntimeError("DD changed the classical-parallel measurement map.")
                prefix = _shared_cut_prefix(prepared, r)
                prefix.metadata.update(
                    preparation_schedule=_CLASSICAL_PARALLEL_VERSION,
                    checkpoint_construction="independently_scheduled_logical_round_count",
                    physical_prefixes_verified=False,
                    routing_sha256=digest)
                family["prefixes"].setdefault(policy, {})[r] = prefix
                family["stats"].setdefault(policy, {})[r] = {"dd_cycles": cycles}
                family["ancestors"].setdefault(policy, {})[r] = {str(r): _shared_digest(prefix)}
                if r == MAX_ROUNDS:
                    family["masters"][policy] = prepared
            except Exception as exc:
                if policy == "none":
                    raise
                failed_policies.add(policy)
                for section in ("masters", "prefixes", "stats", "ancestors"):
                    family[section].pop(policy, None)
                family["failures"].append(dict(
                    stage="policy_preparation", policy=policy, round=r,
                    error_type=type(exc).__name__, error_message=str(exc)))
    family["master_sha256"] = _shared_digest(family["masters"]["none"])
    family["routing_master_sha256"] = family["routing_sha256_by_round"][MAX_ROUNDS]
    return family


def _classical_parallel_timing(circuit, backend, round_number):
    """Audit scheduled critical-path time (not a hardware fidelity prediction)."""
    durations = backend.target.durations()
    clocks = {q: 0 for q in range(circuit.num_qubits)}
    measurements = []
    boundary_time = None
    for item in circuit.data:
        qargs = [circuit.find_bit(q).index for q in item.qubits]
        start = max((clocks[q] for q in qargs), default=0)
        if item.operation.name == "barrier":
            stop = start
            if getattr(item.operation, "label", None) == f"CUMULATIVE_ROUND_{round_number}":
                boundary_time = int(stop)
        elif item.operation.name == "delay":
            if item.operation.unit != "dt":
                raise ValueError("Timing audit expects scheduled delays in dt.")
            stop = start + int(item.operation.duration)
        else:
            duration = durations.get(item.operation.name, qargs, unit="dt")
            stop = start + int(round(duration))
        for q in qargs:
            clocks[q] = stop
        if item.operation.name == "measure_2" and item.clbits:
            bit = circuit.find_bit(item.clbits[0]).index
            if bit < 4 * round_number:
                measurements.append({"classical_bit": bit, "physical_qubit": qargs[0],
                                     "start_dt": int(start), "stop_dt": int(stop)})
    return {"preparation_duration_dt": boundary_time,
            "dt_seconds": float(backend.dt),
            "syndrome_measurement_windows": measurements,
            "bell_measurements_share_start": len({m["start_dt"] for m in measurements}) <= 1,
            "offline_schedule_audit_not_fidelity": True}


def _classical_parallel_write_audit(family):
    """Preserve every independently compiled checkpoint in the existing audit folder."""
    folder = OUTPUT_DIR / "audit"
    folder.mkdir(parents=True, exist_ok=True)
    stem = f"classical_parallel_level_{OPTIMIZATION_LEVEL}_{family['master_sha256'][:12]}"
    for policy, rounds in family["prefixes"].items():
        for r, circuit in rounds.items():
            with (folder / f"{stem}_{policy}_R{r}.qpy").open("wb") as stream:
                qpy.dump(circuit, stream)
    payload = {
        "version": family["version"], "optimization_level": OPTIMIZATION_LEVEL,
        "correction_method": "classical", "initial_physical_layout": family["initial_physical_layout"],
        "master_sha256": family["master_sha256"],
        "routing_master_sha256": family["routing_master_sha256"],
        "routing_sha256_by_round": family["routing_sha256_by_round"],
        "recovery_strategy": family["recovery_strategy"],
        "preflight_complete": family["preflight_complete"],
        "unavailable_preparation_conditions": family["failures"],
        "physical_pairs_by_round": family["pairs"],
        "measurement_maps_by_round": family["mappings"],
        "checkpoint_sha256_by_policy_and_round": family["prefix_sha256"],
        "dd_cycles_by_checkpoint": family["stats"], "timing_by_round": family["timing_by_round"],
        "physical_prefixes_verified": False,
        "checkpoint_construction": family["checkpoint_construction"],
        "preparation_shared_across_channels": True,
        "cross_mode_comparison": "classical_only_independently_scheduled_checkpoints",
    }
    (folder / f"{stem}.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")


def _classical_parallel_route_comparison(records):
    """Check each round's own cached route across levels, not R0 against R3."""
    groups = []
    for _, group in records.groupby(["domain", "round"], sort=False):
        group = group.copy()
        hashes = group["routing_master_sha256"]
        if group["optimization_level"].dropna().nunique() < 2:
            verified, status = pd.NA, "unverified_fewer_than_two_acquired_levels"
        elif hashes.isna().any() or hashes.astype(str).str.len().eq(0).any():
            verified, status = pd.NA, "unverified_missing_route_hash"
        else:
            verified = bool(hashes.nunique() == 1)
            status = ("identical_checkpoint_route_across_acquired_levels" if verified
                      else "different_checkpoint_route_across_acquired_levels")
        group["routing_frozen_across_levels"] = verified
        group["routing_comparison_status"] = status
        groups.append(group)
    return pd.concat(groups, ignore_index=True)



## R4/R5 compilation, audits and hardware-only collection


In [ ]:
# Shared-preparation repair. Definitions only; hardware is submitted by the sweep cell.
import hashlib
import io
from itertools import product
from uuid import uuid4
from qiskit import qpy

_SHARED_BACKEND = None
_SHARED_LAYOUT = None
_SHARED_PREFIX_CACHE = {}
_SHARED_ROUTED_CACHE = {}
_SHARED_AUDIT_ROWS = []
# New results must not overwrite or be mixed with the earlier acquisition.
OUTPUT_DIR = OUTPUT_BASE_DIR / CORRECTION_METHOD
MCM_DD_AUDIT_CSV = OUTPUT_DIR / 'legacy_dd_audit.csv'
EXPERIMENT_VARIANT = 'Classical-only overlap-enabled MCM cascade; R4/R5'


def _shared_digest(circuit):
    """Stable instruction hash: QPY serialization itself contains varying identifiers."""
    def describe(block):
        rows = []
        for item in block.data:
            op = item.operation
            branches = tuple(getattr(op, 'blocks', ()))
            condition = getattr(op, 'condition', None)
            if isinstance(condition, tuple):
                variable, value = condition
                try:
                    variable = ['clbit', block.find_bit(variable).index]
                except Exception:
                    variable = repr(variable)
                condition = [variable, int(value)]
            rows.append(dict(name=op.name, qubits=[block.find_bit(q).index for q in item.qubits],
                             clbits=[block.find_bit(c).index for c in item.clbits],
                             parameters=[] if branches else [str(p) for p in op.params],
                             condition=condition, label=getattr(op, 'label', None),
                             delay_unit=getattr(op, 'unit', None),
                             branches=[describe(branch) for branch in branches]))
        return dict(num_qubits=block.num_qubits, num_clbits=block.num_clbits,
                    global_phase=str(block.global_phase), instructions=rows)
    payload = json.dumps(describe(circuit), sort_keys=True, default=str).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def _shared_counts(circuit):
    counts = {}
    for item in circuit.data:
        name = item.operation.name
        counts[name] = counts.get(name, 0) + 1
        for block in getattr(item.operation, 'blocks', ()):
            for key, value in _shared_counts(block).items():
                counts[key] = counts.get(key, 0) + value
    return counts


_SHARED_CUMULATIVE_VERSION = _CLASSICAL_PARALLEL_VERSION
_SHARED_CUMULATIVE_CACHE = {}






def _shared_syndrome_map(circuit, count):
    measured = {}
    for item in circuit.data:
        if item.operation.name in ('measure', 'measure_2'):
            bit = circuit.find_bit(item.clbits[0]).index
            if bit < count:
                if bit in measured:
                    raise RuntimeError('A syndrome bit was measured more than once.')
                measured[bit] = circuit.find_bit(item.qubits[0]).index
    if set(measured) != set(range(count)):
        raise RuntimeError('Incomplete cumulative syndrome measurement map.')
    return [measured[i] for i in range(count)]


def _shared_cut_prefix(master, round_number):
    """Copy an existing physical prefix verbatim. Never transpile or reschedule it."""
    label = f'CUMULATIVE_ROUND_{round_number}'
    cuts = [i for i, item in enumerate(master.data)
            if item.operation.name == 'barrier' and item.operation.label == label]
    if len(cuts) != 1:
        raise RuntimeError(f'Expected one preserved boundary {label}.')
    registers = master.cregs[:2 * round_number]
    expected_names = [f'r{r}_m{channel}' for r in range(1, round_number + 1)
                      for channel in ('A', 'B')]
    if [reg.name for reg in registers] != expected_names:
        raise RuntimeError('Unexpected cumulative syndrome-register order.')
    prefix = QuantumCircuit(*master.qregs, *registers,
                            name=f'cumulative_preparation_round_{round_number}')
    prefix.global_phase = master.global_phase
    prefix.metadata = copy.deepcopy(master.metadata or {})
    prefix.metadata.update(teleportation_rounds=round_number,
                           shared_compilation_scheme=_SHARED_CUMULATIVE_VERSION)
    for item in master.data[:cuts[0] + 1]:
        cindices = [master.find_bit(c).index for c in item.clbits]
        if any(c >= prefix.num_clbits for c in cindices):
            raise RuntimeError('A later-round operation crossed the prefix boundary.')
        prefix.append(item.operation,
                      [prefix.qubits[master.find_bit(q).index] for q in item.qubits],
                      [prefix.clbits[c] for c in cindices])
    if _shared_counts(prefix).get('measure_2', 0) != 4 * round_number:
        raise RuntimeError('Unexpected syndrome count in cumulative prefix.')
    return prefix




def _shared_active_qubits(circuit, physical_indices=None):
    """Control-flow qargs can include idle device wires; inspect actual leaves."""
    physical_indices = (list(range(circuit.num_qubits)) if physical_indices is None
                        else physical_indices)
    active = set()
    for item in circuit.data:
        physical = [physical_indices[circuit.find_bit(q).index] for q in item.qubits]
        branches = getattr(item.operation, 'blocks', ())
        if branches:
            for branch in branches:
                active.update(_shared_active_qubits(branch, physical))
        elif item.operation.name not in ('barrier', 'delay'):
            active.update(physical)
    return sorted(active)




def _shared_compile_candidate(backend, full_layout, strategy):
    if CORRECTION_METHOD != 'classical':
        raise ValueError('Only classical post-processing is enabled.')
    return _classical_parallel_compile_family(backend, full_layout, strategy)


def _shared_write_cumulative_audit(family):
    return _classical_parallel_write_audit(family)


def _shared_attach_analysis(prefix, suffix, backend, physical_pair, seed_transpiler=None):
    """Compatibility entry point for offline checks and individual analyzers."""
    analyzer = _shared_compact_analyzer(
        backend, suffix, physical_pair,
        SEED_TRANSPILER if seed_transpiler is None else seed_transpiler)
    return _shared_join_analysis(prefix, analyzer, physical_pair, backend)


def _shared_pair_distribution(distribution, width):
    """Marginalize syndrome outcomes without postselection; last two bits are analysis."""
    pair = {}
    for key, value in distribution.items():
        bits = normalize_bit_key(key, width)
        analysis = bits[:2]
        pair[analysis] = pair.get(analysis, 0.0) + float(value)
    return pair


def _shared_fit_tomography(qst, circuits, pair_distributions, shots):
    # Reuse Qiskit Experiments' analysis and the metadata generated by its basis circuits.
    # target=None below prevents automatic ideal-state simulation.
    data = ExperimentData(experiment=qst)
    for basis_circuit, distribution in zip(circuits, pair_distributions):
        counts = probabilities_to_integer_counts(distribution, shots)
        data.add_data({'counts': counts, 'shots': int(shots),
                       'metadata': copy.deepcopy(basis_circuit.metadata)})
    qst.analysis.run(data, replace_results=True).block_for_results()
    result = data.analysis_results(dataframe=True)
    states = result[result['name'] == 'state']
    if len(states) != 1:
        raise RuntimeError(f'Expected one unconditional tomography state, got {len(states)}.')
    return np.asarray(states.iloc[0]['value'].data, dtype=complex)


def _shared_measure_pair(backend, sampler, mitigator, family, channel, policy, label, round_number, domain='Hardware'):
    plan, execution_circuits = _shared_execution_circuits(
        backend, family, policy, round_number, channel, domain)
    prefix = family['prefixes']['none'][round_number]
    mapping = family['mappings'][round_number][channel]
    stats = family['stats'][policy][round_number]
    common_digest = family['prefix_sha256']['none'][round_number]
    prepared_digest = plan['prepared_sha256']
    qst, tomo_suffixes = plan['qst'], plan['tomo_suffixes']
    n_circuits = len(execution_circuits)
    audit_id = uuid4().hex[:12]
    folder = OUTPUT_DIR / 'audit'
    folder.mkdir(parents=True, exist_ok=True)
    with (folder / f'{audit_id}_circuits.qpy').open('wb') as stream:
        qpy.dump(execution_circuits, stream)
    with (folder / f'{audit_id}_common_prefix.qpy').open('wb') as stream:
        qpy.dump(prefix, stream)
    row = dict(audit_id=audit_id, label=label, domain=domain, optimization_level=OPTIMIZATION_LEVEL,
               seed_transpiler=family['recovery_strategy']['seed_transpiler'], dd_policy=policy, round=round_number,
               common_prefix_sha256=common_digest, prepared_prefix_sha256=prepared_digest,
               identical_preparation_across_all_bases=True,
               identical_preparation_across_channels=True,
               shared_compilation_scheme=family['version'],
               cumulative_master_sha256=family['master_sha256'],
               cumulative_prepared_master_sha256=(family['prepared_master_sha256'][policy]
                   if family.get('physical_prefixes_verified', True) else None),
               checkpoint_construction=family.get('checkpoint_construction', 'strict_nested_physical_prefix'),
               preparation_schedule=family.get('preparation_schedule', _CLASSICAL_PARALLEL_VERSION),
               preparation_duration_dt=family.get('timing_by_round', {}).get(round_number, {}).get('preparation_duration_dt'),
               physical_prefixes_verified=family.get('physical_prefixes_verified', True),
               round_boundary=f'CUMULATIVE_ROUND_{round_number}',
               ancestor_prefix_sha256=json.dumps(family['ancestors'][policy][round_number]),
               measurement_physical_map=json.dumps(mapping), dd_cycles=stats['dd_cycles'],
               n_circuits=n_circuits, correction_method=CORRECTION_METHOD, analysis_optimization_level=0,
               routing_optimization_level=0, routing_frozen_across_levels=None, routing_comparison_status='not_yet_compared',
               routing_master_sha256=family.get('routing_sha256_by_round', {}).get(
                                      round_number, family['routing_master_sha256']),
               preparation_attempt=family['recovery_strategy']['attempt'],
               dd_scheduling_method=family['recovery_strategy']['scheduling_method'],
               analysis_is_outside_DD=True, scheduled_no_DD_baseline=True)
    _SHARED_AUDIT_ROWS.append(row)
    pd.DataFrame(_SHARED_AUDIT_ROWS).to_csv(folder / 'audit.csv', index=False)
    print(f"[{label}] {len(execution_circuits)} bases share preparation {prepared_digest[:12]}; DD cycles={stats['dd_cycles']}")
    # Plain Runtime sampler: prepared circuits must NOT enter the old DD wrapper again.
    result = run_sampler_job_with_retries(sampler, execution_circuits, int(SHOTS), label)
    names = [reg.name for reg in execution_circuits[0].cregs]
    counts = sampler_joint_dynamic_counts(result, names)
    # Circuits and per-shot Runtime results are no longer needed for fitting.
    del execution_circuits, result
    (folder / f'{audit_id}_counts.json').write_text(json.dumps({
        'audit': row, 'joint_counts': counts,
        'tomography_metadata': [c.metadata for c in tomo_suffixes],
    }, default=str), encoding='utf-8')
    if len(counts) != n_circuits:
        raise RuntimeError('Unexpected number of measurement results; received counts saved.')
    width = len(mapping)
    rhos, checks = {}, {}
    for method in ('raw', 'm3'):
        if method == 'm3' and mitigator is None:
            # Calibration failed; retain real raw data without inventing mitigated data.
            continue
        try:
            pair_distributions = _tomography_pair_distributions(
                counts, mapping, mitigator, tomo_suffixes, channel, round_number,
                label + f' [audit {audit_id}]', method)
            rho = _shared_fit_tomography(qst, tomo_suffixes, pair_distributions, int(SHOTS))
            fidelity = metrics_from_density_matrix(rho)
            checks[method] = dict(shared_audit_id=audit_id, shared_prefix_sha256=prepared_digest,
                                  preparation_optimization_level=OPTIMIZATION_LEVEL,
                                  shared_compilation_scheme=family['version'],
                                  cumulative_master_sha256=family['master_sha256'],
                                  physical_prefixes_verified=family.get('physical_prefixes_verified', True),
                                  identical_preparation_across_channels=True,
                                  checkpoint_construction=family.get('checkpoint_construction', 'strict_nested_physical_prefix'),
                                  preparation_schedule=family.get('preparation_schedule', _CLASSICAL_PARALLEL_VERSION),
                                  preparation_duration_dt=family.get('timing_by_round', {}).get(round_number, {}).get('preparation_duration_dt'),
                                  ancestor_prefix_sha256=json.dumps(family['ancestors'][policy][round_number]),
                                  analysis_physical_qubits=json.dumps(mapping[-2:]),
                                  routing_optimization_level=0, routing_frozen_across_levels=None, routing_comparison_status='not_yet_compared',
                                  analysis_optimization_level=0, shared_dd_cycles=stats['dd_cycles'],
                                  dd_implementation=('classical_parallel_local_dd'
                                      if family.get('preparation_schedule') == _CLASSICAL_PARALLEL_VERSION
                                      else 'cumulative_master_local_dd'), dd_scheduling_method=family['recovery_strategy']['scheduling_method'],
                                  seed_transpiler=family['recovery_strategy']['seed_transpiler'],
                                  preparation_attempt=family['recovery_strategy']['attempt'],
                                  routing_master_sha256=family.get('routing_sha256_by_round', {}).get(
                                      round_number, family['routing_master_sha256']),
                                  acquisition_domain=domain, correction_method=CORRECTION_METHOD)
            print(f'[{CORRECTION_METHOD}, {method}] F={fidelity:.4f}')
            rhos[method] = rho
        except Exception as exc:
            checks.pop(method, None)
            _shared_failure('reconstruction', exc, channel=channel, round=round_number,
                            policy=policy, domain=domain, method=method, label=label,
                            counts_file=str(folder / f'{audit_id}_counts.json'))
    if not rhos:
        raise RuntimeError('Both reconstructions failed; raw counts saved; no job resubmitted.')
    return rhos, checks



"""Hardware-only sweep cell embedded in both standalone notebooks."""


def run_classical_checkpoint_level_sweep():
    """Collect hardware R4/R5; retain raw data if M3 reconstruction fails."""
    global _SHARED_BACKEND, _SHARED_LAYOUT
    if not RUN_EXPERIMENT:
        raise RuntimeError('Hardware acquisition disabled; set RUN_EXPERIMENT=True explicitly.')
    require_runtime_043()
    records = []
    _SHARED_RECORDS_BY_LEVEL[OPTIMIZATION_LEVEL] = records
    _SHARED_PLANNED_CONDITIONS_BY_LEVEL[OPTIMIZATION_LEVEL] = (
        int(N_TRIALS) * 2 * len(DD_POLICIES) * len(CHECKPOINT_ROUNDS))
    if CASCADE_SCHEME != 'MCM':
        raise ValueError('This notebook is for the MCM cascade.')
    if _SHARED_BACKEND is None:
        _SHARED_BACKEND = make_runtime_service().backend(BACKEND_NAME)
        validate_runtime_backend(_SHARED_BACKEND, require_measure_2=True, wait_for_active=True)
    if backend_qubits := getattr(_SHARED_BACKEND, 'num_qubits', None):
        if backend_qubits < N_LOGICAL_QUBITS:
            raise ValueError('Backend has fewer than 29 qubits; no hardware jobs submitted.')
    if _SHARED_LAYOUT is None:
        _SHARED_LAYOUT = select_balanced_physical_layout(
            build_parallel_classical_cascade(MAX_ROUNDS), _SHARED_BACKEND)
    backend, layout = _SHARED_BACKEND, _SHARED_LAYOUT
    if backend.name != BACKEND_NAME:
        raise ValueError('Backend changed mid-sweep; restart the kernel to choose a new layout.')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    try:
        append_hardware_error_history(backend, HARDWARE_CALIBRATION_HISTORY_PATH)
    except Exception as exc:
        _shared_failure('calibration_history_export', exc)
    print('Initial layout (actual routed measurement qubits are recorded in the shared audit):', layout)

    domain = 'Hardware'
    family = _shared_cumulative_family(backend, layout)
    if family is None or not family['conditions']:
        _shared_failure('domain_unavailable', RuntimeError('No valid preflight conditions.'), domain=domain)
        _shared_save_progress(records)
        return _shared_records_frame(records)
    _shared_write_cumulative_audit(family)
    calibrated_qubits = set()
    for policy, r, channel in family['conditions']:
        calibrated_qubits.update(family['mappings'][r][channel])
    try:
        sampler = make_hardware_session_sampler(backend)
        sampler.options.dynamical_decoupling.enable = False
    except Exception as exc:
        _shared_failure('domain_setup', exc, domain=domain)
        _shared_save_progress(records)
        return _shared_records_frame(records)
    try:
        mitigator = calibrate_m3(
            backend, sampler, sorted(calibrated_qubits), shots=M3_CALIBRATION_SHOTS)
        if mitigator is None:
            raise RuntimeError('Calibration returned no M3 mitigator.')
    except Exception as exc:
        _shared_failure('m3_calibration', exc, domain=domain, raw_acquisition_available=True)
        mitigator = None
    mitigator = _m3_recovery_state(backend, sampler, mitigator, domain)

    previous = {}
    for trial in range(1, N_TRIALS + 1):
        for channel in ('A', 'B'):
            for r in CHECKPOINT_ROUNDS:
                policies = list(DD_POLICIES)
                offset = (trial - 1 + r + (channel == 'B')) % len(policies)
                policies = policies[offset:] + policies[:offset]
                for policy in policies:
                    # Do not substitute another policy or prefix for a failed
                    # compilation condition. Failures are already audited.
                    if (policy, r, channel) not in family['conditions']:
                        continue
                    strategy = family['recovery_strategy']
                    label = (f'{domain}, {CORRECTION_METHOD}, shared-preparation, '
                             f'level={OPTIMIZATION_LEVEL}, trial={trial}, '
                             f'channel={channel}, round={r}, policy={policy}')
                    context = dict(domain=domain, trial=trial, channel=channel,
                                   round=r, policy=policy, label=label)
                    try:
                        rhos, checks = _shared_measure_pair(
                            backend, sampler, mitigator, family, channel,
                            policy, label, r, domain=domain)
                    except Exception as exc:
                        # Job handling belongs to the measurement helper; this
                        # catch never recompiles or resubmits the condition.
                        _shared_failure('condition_acquisition', exc, **context)
                        _shared_save_progress(records)
                        continue
                    for method, rho in rhos.items():
                        try:
                            key = (domain, trial, channel, policy, method, r)
                            input_rho = (rho if r == 0 else previous.get(
                                (domain, trial, channel, policy, method, r - 1)))
                            available = input_rho is not None
                            source = ('measured_source_EPR_pair' if r <= 1
                                      else f'measured_round_{r - 1}_output')
                            if not available:
                                source = f'unmeasured_round_{r - 1}_output' if r - 1 not in CHECKPOINT_ROUNDS else f'unavailable_measured_round_{r - 1}_output'
                            state_pair = copy.deepcopy(trajectory_state_pair_spec(layout, channel, r))
                            state_pair['analysis_circuit_logical_indices'] = list(
                                state_pair['full_circuit_logical_indices'])
                            state_pair['state_physical_qubits'] = json.loads(
                                checks[method]['analysis_physical_qubits'])
                            pending = []
                            # Saves both rho_real and rho_imag, plus the input
                            # state payload, for EACH successful raw/M3 result
                            # in the classical-only correction mode.
                            append_mcm_cascade_record(
                                pending, trial, method, channel, r, rho,
                                policy, domain=domain, input_rho=input_rho,
                                input_state_source=source, state_pair=state_pair,
                                input_state_available=available)
                            record = pending[0]
                            record.update(checks[method])
                            record.update(
                                optimization_level=OPTIMIZATION_LEVEL,
                                series_key=f'{CORRECTION_METHOD}_hardware_{policy}_{method}',
                                dd_enabled=bool(policy != 'none'),
                                dd_sequence=policy if policy != 'none' else 'none',
                                dd_implementation=checks[method]['dd_implementation'],
                                dd_scheduling_method=strategy['scheduling_method'],
                                requested_seed_transpiler=int(SEED_TRANSPILER),
                                effective_seed_transpiler=int(strategy['seed_transpiler']),
                                seed_transpiler=int(strategy['seed_transpiler']),
                                recovery_attempt=int(strategy['attempt']),
                                routing_master_sha256=checks[method]['routing_master_sha256'],
                                routing_frozen_across_levels=pd.NA,
                                routing_comparison_status='pending_acquired_level_comparison',
                                round0_dd_cycles=np.nan, round0_dd_status=None,
                                server_builtin_dd_enabled=False, execution_target='QPU')
                            if r == 0:
                                cycles = checks[method]['shared_dd_cycles']
                                record['round0_dd_cycles'] = cycles
                                record['round0_dd_status'] = (
                                    'disabled' if policy == 'none' else
                                    'inserted' if cycles else 'no_eligible_idle_window')
                            records.append(record)
                            previous[key] = rho
                        except Exception as exc:
                            _shared_failure('record_processing', exc, method=method, **context)
                    _shared_save_progress(records)
    _shared_save_progress(records)
    return _shared_records_frame(records)


print('Classical R4/R5 loaded: independent overlap-enabled checkpoints, nine tomography bases, full cumulative Pauli frames.')


_SHARED_RECOVERY_EVENTS = []
_SHARED_FAILURES = []
_SHARED_RECORDS_BY_LEVEL = {}


def _shared_recovery_event(stage, exc=None, **context):
    """Keep recovered attempts separate from conditions that could not be acquired."""
    from datetime import datetime, timezone
    event = dict(timestamp=datetime.now(timezone.utc).isoformat(), stage=stage,
                 level=OPTIMIZATION_LEVEL, correction_method=CORRECTION_METHOD)
    event.update(context)
    if exc is not None:
        event.update(error_type=type(exc).__name__, error_message=str(exc))
    _SHARED_RECOVERY_EVENTS.append(event)
    try:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        with (OUTPUT_DIR / 'recovery_events.jsonl').open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, default=str) + '\n')
    except OSError as log_error:
        print(f'Could not persist recovery log: {log_error}', flush=True)
    print(f'[{stage}] {context} {str(exc) if exc is not None else ""}', flush=True)
    return event


def _shared_failure(stage, exc=None, **context):
    event = _shared_recovery_event(stage, exc, **context)
    _SHARED_FAILURES.append(event)
    return event


def _shared_recovery_strategies():
    # The requested level, physical layout and DD policy never change silently.
    return [dict(attempt=i + 1, seed_transpiler=int(SEED_TRANSPILER) + offset,
                 scheduling_method=schedule)
            for i, (offset, schedule) in enumerate(
                [(0, 'alap'), (101, 'alap'), (202, 'asap'), (303, 'asap')])]


def _shared_build_condition(backend, family, policy, round_number, channel):
    """Validate full circuits one at a time; retain only compact analyzers."""
    prepared = family['prefixes'][policy][round_number]
    mapping = tuple(family['mappings'][round_number][channel])
    physical_pair = mapping[-2:]
    bundle = _shared_analysis_bundle(family)
    suffixes = bundle['suffixes']  # Exactly nine tomography settings.
    cache = family['_analysis_cache']
    analyzers = []
    seed = family['recovery_strategy']['seed_transpiler']
    for basis_index, suffix in enumerate(suffixes):
        key = (physical_pair, basis_index, seed)
        if key not in cache:
            cache[key] = _shared_compact_analyzer(backend, suffix, physical_pair, seed)
        analyzer = cache[key]
        # Complete preflight still occurs before any calibration or experiment.
        # The temporary full circuit is released before assembling the next one.
        checked = _shared_join_analysis(prepared, analyzer, physical_pair, backend)
        del checked
        analyzers.append(analyzer)
    return dict(
        qst=bundle['qst'], tomo_suffixes=bundle['tomo_suffixes'],
        analyzers=tuple(analyzers), measurement_map=mapping,
        prepared_sha256=family['prefix_sha256'][policy][round_number],
        analyzer_sha256=tuple(_shared_digest(c) for c in analyzers))


def _shared_preflight_candidate(backend, family):
    """Snapshot the verified prefixes and check all conditions with bounded RAM."""
    family['conditions'] = {}
    family['prefix_sha256'] = {
        policy: {r: _shared_digest(circuit) for r, circuit in rounds.items()}
        for policy, rounds in family['prefixes'].items()}
    family['prepared_master_sha256'] = {
        policy: _shared_digest(master) for policy, master in family['masters'].items()}
    for policy in DD_POLICIES:
        if policy not in family['prefixes']:
            continue
        for round_number in CHECKPOINT_ROUNDS:
            for channel in ('A', 'B'):
                try:
                    plan = _shared_build_condition(backend, family, policy, round_number, channel)
                    family['conditions'][(policy, round_number, channel)] = plan
                except Exception as exc:
                    family['failures'].append(dict(
                        stage='analysis_preflight', policy=policy, round=round_number,
                        channel=channel, error_type=type(exc).__name__, error_message=str(exc)))
    return family


def _shared_verify_plan(family, policy, round_number, channel):
    """Check stored prefixes and analyzers; full circuits do not remain in RAM."""
    plan = family['conditions'][(policy, round_number, channel)]
    expected_prefixes = family['prefix_sha256'][policy]
    # Check only acquired checkpoints; they are independently scheduled, not
    # strict nested physical prefixes. Snapshots detect later mutation.
    for earlier in (r for r in CHECKPOINT_ROUNDS if r <= round_number):
        prefix = family['prefixes'][policy][earlier]
        if _shared_digest(prefix) != expected_prefixes[earlier]:
            raise RuntimeError('Selected preparation changed after preflight.')
    if plan['prepared_sha256'] != expected_prefixes[round_number]:
        raise RuntimeError('Analysis plan refers to a different preparation.')
    if tuple(family['mappings'][round_number][channel]) != plan['measurement_map']:
        raise RuntimeError('Selected measurement map changed after preflight.')
    if len(plan['analyzers']) != len(plan['analyzer_sha256']):
        raise RuntimeError('An analysis basis was removed after preflight.')
    for analyzer, digest in zip(plan['analyzers'], plan['analyzer_sha256']):
        if _shared_digest(analyzer) != digest:
            raise RuntimeError('An analysis basis changed after preflight.')
    return plan


def _shared_cumulative_family(backend, full_layout):
    """Initial compilation + three changed strategies; select one family only."""
    if (len(full_layout) != N_LOGICAL_QUBITS or len(set(full_layout)) != N_LOGICAL_QUBITS
            or set(DD_POLICIES) != {'none', 'XX', 'XY4'}):
        _shared_failure('preparation_configuration', ValueError(
            'Need 29 distinct physical qubits and policies none/XX/XY4. '
            'Routing retries cannot repair this configuration.'))
        return None
    best = None
    expected = len(DD_POLICIES) * len(CHECKPOINT_ROUNDS) * 2
    for strategy in _shared_recovery_strategies():
        if strategy['attempt'] > 1:
            # A failed cache entry must never contaminate a changed attempt.
            _SHARED_CUMULATIVE_CACHE.clear()
            _SHARED_ROUTED_CACHE.clear()
        try:
            candidate = _shared_compile_candidate(backend, full_layout, strategy)
            candidate = _shared_preflight_candidate(backend, candidate)
            available = len(candidate['conditions'])
            _shared_recovery_event('preparation_attempt', strategy=strategy,
                                   valid_conditions=available, expected_conditions=expected,
                                   preparation_failures=candidate['failures'])
            if available and (best is None or available > len(best['conditions'])):
                best = candidate
            if available == expected:
                break
        except Exception as exc:
            _shared_recovery_event('preparation_attempt_failed', exc, strategy=strategy)
    # Persist the selected family before any acquisition; do not return invalid candidates.
    if best is None:
        _shared_failure('preparation_exhausted', RuntimeError(
            'No valid preparation after initial attempt plus three recovery attempts.'))
        return None
    best['preflight_complete'] = len(best['conditions']) == expected
    for failure in best['failures']:
        details = dict(failure)
        stage = details.pop('stage')
        _shared_failure(stage, selected_attempt=best['recovery_strategy']['attempt'], **details)
    _shared_recovery_event('preparation_selected', strategy=best['recovery_strategy'],
        valid_conditions=len(best['conditions']), expected_conditions=expected,
        master_sha256=best['master_sha256'], complete=best['preflight_complete'])
    return best


_SHARED_PLANNED_CONDITIONS_BY_LEVEL = {}


def _shared_empty_records():
    """Keep downstream exports and filters valid when a level has no usable rows."""
    return pd.DataFrame(columns=[
        'optimization_level', 'domain', 'trial', 'channel', 'round', 'dd_policy',
        'method', 'series_key', 'fidelity', 'correction_method',
        'rho_real', 'rho_imag', 'input_rho_real', 'input_rho_imag',
        'input_state_available', 'input_state_source', 'shared_audit_id',
        'shared_prefix_sha256', 'routing_master_sha256',
        'requested_seed_transpiler', 'effective_seed_transpiler',
        'dd_scheduling_method', 'recovery_attempt', 'round0_dd_cycles', 'round0_dd_status',
        'routing_frozen_across_levels',
        'routing_comparison_status'])


def _shared_records_frame(records):
    frame = records.copy() if isinstance(records, pd.DataFrame) else pd.DataFrame(records)
    return frame if not frame.empty else _shared_empty_records()


def _shared_update_status(records, planned_conditions):
    """An acquired condition can still be partial if one reconstruction is absent."""
    from datetime import datetime, timezone
    frame = _shared_records_frame(records)
    planned = int(planned_conditions)
    keys = ['optimization_level', 'domain', 'trial', 'channel', 'round', 'dd_policy']
    completed = len(frame[keys].drop_duplicates()) if len(frame) else 0
    method_rows = len(frame[keys + ['method']].drop_duplicates()) if len(frame) else 0
    payload = dict(
        updated_at=datetime.now(timezone.utc).isoformat(),
        status=('no_valid_data' if frame.empty else
                'complete' if completed == planned and method_rows == 2 * planned
                and not _SHARED_FAILURES else 'partial'),
        planned_conditions=planned, completed_conditions=completed,
        missing_conditions=max(0, planned - completed),
        expected_method_rows=2 * planned, saved_method_rows=method_rows,
        missing_method_rows=max(0, 2 * planned - method_rows),
        saved_rows=len(frame), valid_records=len(frame), failure_count=len(_SHARED_FAILURES),
        failures=list(_SHARED_FAILURES),
        recovery_event_count=len(_SHARED_RECOVERY_EVENTS))
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    temporary = OUTPUT_DIR / 'acquisition_status.json.tmp'
    temporary.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')
    temporary.replace(OUTPUT_DIR / 'acquisition_status.json')
    return payload


def _shared_save_progress(records):
    """Export errors do not discard the live rows held for final level export."""
    try:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        _shared_records_frame(records).to_csv(
            OUTPUT_DIR / f'shared_level_{OPTIMIZATION_LEVEL}_checkpoint.csv', index=False)
    except Exception as exc:
        _shared_failure('checkpoint_export', exc)
    try:
        live = [row for rows in _SHARED_RECORDS_BY_LEVEL.values() for row in rows]
        _shared_update_status(live, sum(_SHARED_PLANNED_CONDITIONS_BY_LEVEL.values()))
    except Exception as exc:
        _shared_failure('status_export', exc)


def _shared_route_comparison_group(records):
    """Verify routing identity only from the routes used in acquired levels."""
    frame = _shared_records_frame(records)
    if (not frame.empty and 'preparation_schedule' in frame
            and frame['preparation_schedule'].eq(_CLASSICAL_PARALLEL_VERSION).all()):
        return _classical_parallel_route_comparison(frame)
    levels = frame['optimization_level'].dropna().unique()
    hashes = frame.get('routing_master_sha256', pd.Series(index=frame.index, dtype=object))
    if len(levels) < 2:
        verified, status = pd.NA, 'unverified_fewer_than_two_acquired_levels'
    elif hashes.isna().any() or hashes.astype(str).str.len().eq(0).any():
        verified, status = pd.NA, 'unverified_missing_route_hash'
    else:
        verified = bool(hashes.nunique() == 1)
        status = 'identical_across_acquired_levels' if verified else 'different_across_acquired_levels'
    frame['routing_frozen_across_levels'] = verified
    frame['routing_comparison_status'] = status
    return frame

def _shared_route_comparison(records):
    """Verify routes across levels separately for each correction method."""
    frame = _shared_records_frame(records)
    if frame.empty:
        return _shared_route_comparison_group(frame)
    if not frame['domain'].eq('Hardware').all():
        raise ValueError('This acquisition driver only accepts hardware records.')
    groups = []
    for _, mode in frame.groupby('correction_method', sort=False):
        groups.append(_shared_route_comparison_group(mode))
    return pd.concat(groups, ignore_index=True)



def _shared_analysis_bundle(family):
    """Keep one small basis library per compilation attempt."""
    if '_analysis_bundle' not in family:
        qst = StateTomography(QuantumCircuit(2), target=None)
        qst.set_transpile_options(
            optimization_level=0,
            seed_transpiler=family['recovery_strategy']['seed_transpiler'])
        tomography = tuple(qst.circuits())
        if len(tomography) != 9:
            raise RuntimeError('Expected nine two-qubit Pauli tomography bases.')
        suffixes = list(tomography)
        family['_analysis_bundle'] = dict(
            qst=qst, tomo_suffixes=tomography, suffixes=tuple(suffixes))
        family['_analysis_cache'] = {}
    return family['_analysis_bundle']


def _shared_compact_analyzer(backend, suffix, physical_pair, seed_transpiler):
    """Compile the same analyzer as v2, retaining only its two used wires."""
    compiled = transpile(
        suffix, backend=backend, initial_layout=list(physical_pair),
        optimization_level=0, seed_transpiler=seed_transpiler,
        approximation_degree=1.0)
    physical_to_local = {int(q): i for i, q in enumerate(physical_pair)}
    if len(physical_to_local) != 2 or compiled.num_clbits != 2:
        raise RuntimeError('Analyzer must measure two distinct physical qubits.')
    compact = QuantumCircuit(2, 2, name='compact_native_analyzer')
    compact.global_phase = compiled.global_phase
    for item in compiled.data:
        # Basis rotations and readouts must not route onto another qubit.
        physical = [compiled.find_bit(q).index for q in item.qubits]
        if (any(q not in physical_to_local for q in physical)
                or getattr(item.operation, 'blocks', ())):
            raise RuntimeError('Analyzer compilation used an unexpected wire or control flow.')
        compact.append(item.operation,
                       [compact.qubits[physical_to_local[q]] for q in physical],
                       [compact.clbits[compiled.find_bit(c).index] for c in item.clbits])
    return compact


def _shared_join_analysis(prefix, analyzer, physical_pair, backend, *, validate=True):
    """Append native operations without recompiling or rescheduling preparation."""
    out = prefix.copy()
    analysis = ClassicalRegister(2, 'analysis')
    out.add_register(analysis)
    out.global_phase += analyzer.global_phase
    for item in analyzer.data:
        out.append(item.operation,
                   [out.qubits[physical_pair[analyzer.find_bit(q).index]] for q in item.qubits],
                   [analysis[analyzer.find_bit(c).index] for c in item.clbits])
    if list(out.data[:len(prefix.data)]) != list(prefix.data):
        raise RuntimeError('Appending the analyzer changed the state preparation.')
    if validate:
        _validate_physical_readouts(out, backend)
        assert_terminal_measurements_are_final(out, 'shared-preparation final circuit')
    return out


def _shared_execution_circuits(backend, family, policy, round_number, channel, domain):
    """Materialize one hardware condition with exactly nine analyzers."""
    if domain != 'Hardware':
        raise ValueError('Only hardware acquisition is enabled in this notebook.')
    plan = _shared_verify_plan(family, policy, round_number, channel)
    prepared = family['prefixes'][policy][round_number]
    circuits = []
    for analyzer in plan['analyzers']:
        circuit = _shared_join_analysis(
            prepared, analyzer, plan['measurement_map'][-2:], backend, validate=False)
        circuits.append(circuit)
    return plan, circuits









print('Fidelity-only acquisition loaded; full cumulative Pauli frames for classical correction.')


def _m3_recovery_state(backend, sampler, mitigator, domain):
    """One recovery cache per domain and calibration cycle."""
    return dict(backend=backend, sampler=sampler, original=mitigator,
                domain=domain, recovered={})


def _m3_recovery_note(state, label, **details):
    from datetime import datetime, timezone
    payload = dict(time=datetime.now(timezone.utc).isoformat(),
                   domain=state['domain'], level=OPTIMIZATION_LEVEL,
                   label=label, **details)
    path = OUTPUT_DIR / 'M3_recovery.txt'
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(payload, default=str) + '\n')
    print('[M3 recovery] ' + json.dumps(details, default=str), flush=True)


def _m3_unusable_qubits(mitigator, qubits):
    """Reject missing/nonfinite/singular calibrations and M3's faulty flags."""
    matrices = getattr(mitigator, 'single_qubit_cals', None)
    faulty = set(getattr(mitigator, 'faulty_qubits', []) or [])
    bad = []
    for q in dict.fromkeys(qubits):
        try:
            a = np.asarray(matrices[q], dtype=float)
            valid = (a.shape == (2, 2) and np.isfinite(a).all()
                     and (a >= 0).all() and np.allclose(a.sum(axis=0), 1.0)
                     and np.linalg.cond(a) < 1e8 and q not in faulty)
        except (TypeError, IndexError, ValueError, np.linalg.LinAlgError):
            valid = False
        if not valid:
            bad.append(int(q))
    return bad


def _m3_marginal_counts(counts, width, positions):
    """Sum over excluded bits; positions are in classical-bit order (LSB first)."""
    result = {}
    for key, count in counts.items():
        bits = normalize_bit_key(key, width)[::-1]
        selected = ''.join(bits[i] for i in positions)[::-1]
        result[selected] = result.get(selected, 0) + int(count)
    return result


def _m3_correct_pairs(counts, mapping, mitigator, positions):
    selected = [mapping[i] for i in positions]
    bad = _m3_unusable_qubits(mitigator, selected)
    if bad:
        raise ValueError(f'Unusable M3 calibration for physical qubits {bad}')
    pairs = []
    for distribution in counts:
        reduced = _m3_marginal_counts(distribution, len(mapping), positions)
        corrected = m3_probability_distribution(reduced, selected, mitigator)
        values = np.asarray(list(corrected.values()), dtype=float)
        if (not values.size or not np.isfinite(values).all()
                or np.any(values < -1e-10) or values.sum() <= 0):
            raise ValueError('M3 returned an empty or nonfinite probability distribution')
        pair = _shared_pair_distribution(corrected, len(selected))
        total = sum(pair.values())
        pairs.append({key: float(value / total) for key, value in pair.items()})
    return pairs


def _m3_recover_pair_distributions(counts, mapping, state, label):
    """Original correction plus three distinct recovery attempts, no data resubmission."""
    mapping = list(map(int, mapping))
    key = tuple(mapping)
    pair = mapping[-2:]
    if len(pair) != 2 or pair[0] == pair[1]:
        raise ValueError(f'Invalid physical analysis pair: {pair}')
    pair_positions = [len(mapping)-2, len(mapping)-1]
    cached = state['recovered'].get(key)
    initial = cached or (state['original'], list(range(len(mapping))))
    try:
        result = _m3_correct_pairs(counts, mapping, *initial)
        if cached:
            _m3_recovery_note(state, label, action='reused_recovered_calibration',
                              retained_physical=[mapping[i] for i in cached[1]],
                              excluded_classical_bits=[i for i in range(len(mapping)) if i not in cached[1]])
        return result
    except Exception as exc:
        state['recovered'].pop(key, None)
        _m3_recovery_note(state, label, attempt=0, action='initial_correction_failed',
                          error=str(exc), analysis_pair=pair,
                          unusable_physical=_m3_unusable_qubits(initial[0], mapping))

    actions = ('exclude_unusable_syndrome_bits', 'recalibrate_analysis_pair',
               'calibrate_pair_qubits_individually_with_more_shots')
    for attempt, action in enumerate(actions, 1):
        shots = int(M3_CALIBRATION_SHOTS) * (4 if attempt == 3 else 1)
        _m3_recovery_note(state, label, attempt=attempt, action=action,
                          analysis_pair=pair, calibration_shots_per_prepared_state=shots if attempt > 1 else 0)
        try:
            if attempt == 1:
                mitigator = state['original']
                bad = set(_m3_unusable_qubits(mitigator, mapping))
                if bad.intersection(pair):
                    raise ValueError('Analysis-pair calibration is unusable; pair must be recalibrated')
                positions = [i for i, q in enumerate(mapping) if q not in bad]
                # If no identifiable bad matrix exists, remove the joint syndrome
                # solve that failed numerically and correct the unconditional pair.
                if not bad:
                    positions = pair_positions
            elif attempt == 2:
                mitigator = calibrate_m3(state['backend'], state['sampler'], pair,
                    shots=shots, label_suffix=f'M3 recovery 2: {label}')
                positions = pair_positions
            else:
                matrices = [None] * int(state['backend'].num_qubits)
                errors = []
                for q in pair:
                    try:
                        single = calibrate_m3(state['backend'], state['sampler'], [q],
                            shots=shots, label_suffix=f'M3 recovery 3, qubit {q}: {label}')
                        if _m3_unusable_qubits(single, [q]):
                            raise ValueError(f'Qubit {q} still has unusable calibration')
                        matrices[q] = np.asarray(single.single_qubit_cals[q], dtype=float).copy()
                    except Exception as exc:
                        errors.append(f'q{q}: {exc}')
                if errors:
                    raise RuntimeError('; '.join(errors))
                mitigator = mthree.M3Mitigation(state['backend'])
                mitigator.cals_from_matrices(matrices)
                positions = pair_positions
            recovered = _m3_correct_pairs(counts, mapping, mitigator, positions)
            state['recovered'][key] = (mitigator, positions)
            _m3_recovery_note(state, label, attempt=attempt, action='recovery_succeeded',
                              retained_physical=[mapping[i] for i in positions],
                              excluded_classical_bits=[i for i in range(len(mapping)) if i not in positions],
                              excluded_physical=[mapping[i] for i in range(len(mapping)) if i not in positions],
                              retained_matrices={str(q): np.asarray(mitigator.single_qubit_cals[q]).tolist()
                                                 for q in dict.fromkeys(mapping[i] for i in positions)},
                              shots_discarded=0, plot_method='m3')
            return recovered
        except Exception as exc:
            _m3_recovery_note(state, label, attempt=attempt, action='recovery_failed', error=str(exc))
    raise RuntimeError('M3 failed after three recovery attempts; raw counts remain saved. See M3_recovery.txt.')

## Plotting existing records: bar plots only
One hardware figure per correction method, optimisation level and channel.
As in the original replot section, raw results are stacked above M3 results;
each round groups No DD, XX and XY4 bars. Only bar plots are generated.
Plots use the original output parent: `epr_replots/<timestamp>/`, with
`feedforward/` and `classical/` subfolders. No backend jobs are submitted.

In [ ]:
"""Original-style hardware fidelity bars only; never submit backend jobs."""
import ast
from datetime import datetime


def _plot_ci(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    return (float(values.mean()) if n else np.nan,
            float(student_t.ppf(.995, n - 1) * values.std(ddof=1) / np.sqrt(n))
            if n > 1 else np.nan, n)


def _plot_rho(row):
    """Read the preserved density-matrix exports; plotting does not need them."""
    def array(value):
        return np.asarray(ast.literal_eval(value) if isinstance(value, str) else value, dtype=float)
    rho = array(row["rho_real"]) + 1j * array(row["rho_imag"])
    if rho.shape != (4, 4) or not np.isfinite(rho).all():
        raise ValueError("Expected a finite saved 4x4 density matrix.")
    if not np.allclose(rho, rho.conj().T, atol=1e-7) or not np.isclose(np.trace(rho), 1, atol=1e-7):
        raise ValueError("Saved density matrix is not Hermitian and normalized.")
    eigenvalues, vectors = np.linalg.eigh(rho)
    if eigenvalues.min() < -1e-7:
        raise ValueError("Saved density matrix is not positive semidefinite.")
    eigenvalues = np.maximum(eigenvalues, 0)
    return (vectors * (eigenvalues / eigenvalues.sum())) @ vectors.conj().T


def _plot_slug(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def _plot_ci_title(sample_counts):
    if not np.any(np.asarray(sample_counts, dtype=float) >= 2):
        return "Single trial; confidence interval unavailable"
    return "Pointwise 99% Student-t intervals where n >= 2"


def _save_fidelity_figure(fig, path, show):
    fig.savefig(path, dpi=180, bbox_inches="tight", format="png")
    if show:
        plt.show()
    plt.close(fig)


def replot_existing_sweep(records, output_dir=None, show=True):
    """One bar figure per correction mode, level and channel.

    Match the original replot layout after removing the second metric column:
    Raw above M3; rounds on the x-axis; DD policies grouped within each round.
    Save only fidelity bars and their supporting records/summary CSVs.
    """
    frame = pd.DataFrame(records).copy()
    required = {"correction_method", "domain", "optimization_level", "dd_policy",
                "method", "channel", "round", "trial", "fidelity"}
    if missing := required - set(frame.columns):
        raise ValueError(f"Missing columns: {sorted(missing)}")
    hardware = frame["domain"].eq("Hardware")
    if not hardware.all():
        print(f"Skipping {int((~hardware).sum())} non-hardware rows; only hardware is plotted.")
    selected = hardware & frame["correction_method"].eq("classical") & frame["round"].isin(CHECKPOINT_ROUNDS)
    if int((hardware & ~selected).sum()):
        print("Skipping non-classical or non-R4/R5 hardware rows.")
    frame = frame.loc[selected].copy()
    if frame.empty:
        raise ValueError("No hardware records were supplied.")
    keys = ["correction_method", "domain", "optimization_level", "dd_policy",
            "method", "channel", "round", "trial"]
    if frame[keys].isna().any().any() or frame.duplicated(keys).any():
        raise ValueError("Missing or duplicate condition/trial labels; do not mix repeated runs.")
    if not frame["correction_method"].isin(CORRECTION_METHODS).all():
        raise ValueError("Unknown correction method.")
    if not frame["method"].isin(("raw", "m3")).all():
        raise ValueError("Unknown processing method.")
    if not frame["dd_policy"].isin(("none", "XX", "XY4")).all():
        raise ValueError("Unknown DD policy.")
    # This is the final replot location in the two read-only original scripts.
    out = Path(output_dir) if output_dir is not None else (
        OUTPUT_ROOT / "epr_replots" / datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=True)
    # Keep complete records, including rho_real/rho_imag, even though matrices
    # are no longer used to generate additional state-comparison figures.
    frame.to_csv(out / "all_correction_methods_records_used.csv", index=False)
    summaries = []
    group_keys = keys[:-1]
    for key, group in frame.groupby(group_keys, sort=True):
        mean, ci, n = _plot_ci(group["fidelity"])
        summaries.append(dict(zip(group_keys, key), metric="fidelity",
                              mean=mean, ci99_halfwidth=ci, n_trials=n))
    summary = pd.DataFrame(summaries)
    summary.to_csv(out / "entanglement_fidelity_summary.csv", index=False)
    files = []
    dd_colors = {"none": "#0072B2", "XX": "#009E73", "XY4": "#E69F00"}
    dd_labels = {"none": "No DD", "XX": "XX", "XY4": "XY4"}
    plt.rcParams.update({"axes.titleweight": "normal", "font.weight": "normal"})
    for (correction, domain, level, channel), part in summary.groupby(
            ["correction_method", "domain", "optimization_level", "channel"], sort=True):
        folder = out / correction
        folder.mkdir(parents=True, exist_ok=True)
        part.to_csv(folder / f"{_plot_slug(domain)}_level_{_plot_slug(level)}_channel_{channel}_fidelity.csv",
                    index=False)
        methods = [method for method in ("raw", "m3") if method in set(part["method"])]
        fig, axes = plt.subplots(len(methods), 1, figsize=(12, 4.5 * len(methods)),
                                 squeeze=False, sharex=True, sharey=True)
        upper_tops = []
        for axis, method in zip(axes.flat, methods):
            subset = part[part["method"].eq(method)]
            rounds = sorted(part["round"].unique())
            policies = [policy for policy in ("none", "XX", "XY4")
                        if policy in set(part["dd_policy"])]
            positions = np.arange(len(rounds), dtype=float)
            width = .8 / len(policies)
            for index, policy in enumerate(policies):
                rows = subset[subset["dd_policy"].eq(policy)].set_index("round").reindex(rounds)
                means = rows["mean"].to_numpy(float)
                errors = rows["ci99_halfwidth"].to_numpy(float)
                x = positions + (index - (len(policies) - 1) / 2) * width
                axis.bar(x, means, width, color=dd_colors[policy], label=dd_labels[policy],
                         edgecolor="#333333", linewidth=.7)
                available = np.isfinite(errors) & np.isfinite(means)
                axis.errorbar(x[available], means[available], yerr=errors[available],
                              fmt="none", color="black", capsize=4, capthick=1.2,
                              elinewidth=1.2, zorder=3)
                for xx, value, error in zip(x, means, errors):
                    if np.isfinite(value):
                        top = value + (error if np.isfinite(error) else 0)
                        upper_tops.append(top)
                        axis.annotate(f"{value:.3f}", (xx, top),
                                      xytext=(0, 6), textcoords="offset points",
                                      ha="center", va="bottom", fontsize=11,
                                      fontweight="normal", color="black")
            axis.set_xticks(positions, [f"R{r}" for r in rounds])
            axis.set_ylabel("Entanglement fidelity to |Phi+>")
            axis.set_title("Raw" if method == "raw" else "M3-mitigated", fontweight="normal")
            axis.grid(axis="y", alpha=.25)
            axis.set_axisbelow(True)
            axis.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
            axis.tick_params(axis="x", labelbottom=True)
        upper = max(1.12, max(upper_tops or [0]) + .1)
        for axis in axes.flat:
            axis.set_ylim(0, upper)
            axis.set_xlabel("Teleportation round")
        fixture = ("SYNTHETIC VALIDATION — NOT HARDWARE RESULTS\n"
                   if "data_origin" in frame and
                   part.empty is False and
                   frame["data_origin"].eq("SYNTHETIC_VALIDATION_NOT_HARDWARE").all()
                   else "")
        fig.suptitle(f"{fixture}Rank {DISENTANGLER_RANK} | {domain} | Optimisation level {level} | Channel {channel}\n"
                     f"{CORRECTION_LABELS[correction]} — {_plot_ci_title(part['n_trials'])}",
                     fontweight="normal", fontsize=13)
        fig.tight_layout(rect=(0, 0, 1, .91))
        path = folder / f"{_plot_slug(domain)}_level_{_plot_slug(level)}_channel_{channel}_metrics.png"
        _save_fidelity_figure(fig, path, show)
        files.append(path)
    print(f"Saved {len(files)} PNG bar plots and supporting CSVs to {out.resolve()}")
    print("Used existing records only; no backend jobs submitted. One trial has no estimated CI.")
    return out

## Classical-only R4/R5 acquisition across optimisation levels


In [ ]:
def run_classical_R456_sweep():
    """Acquire classical R4/R5 only; no earlier-endpoint jobs or physical corrections."""
    global CORRECTION_METHOD, OUTPUT_DIR, MCM_DD_AUDIT_CSV, OPTIMIZATION_LEVEL
    global OUTPUT_BASE_DIR, _SHARED_RECORDS_BY_LEVEL, _SHARED_PLANNED_CONDITIONS_BY_LEVEL
    global _SHARED_FAILURES, _SHARED_RECOVERY_EVENTS, _SHARED_AUDIT_ROWS
    if not RUN_EXPERIMENT:
        raise RuntimeError("Hardware acquisition disabled. Set RUN_EXPERIMENT=True explicitly.")
    require_runtime_043()
    from datetime import datetime
    OUTPUT_BASE_DIR = OUTPUT_ROOT / f"mcm_shared_prep_rank{DISENTANGLER_RANK}_classical_R456" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)
    mode_state = {mode: dict(records={}, planned={}, failures=[], events=[], audit=[])
                  for mode in CORRECTION_METHODS}
    acquired = []
    settings = dict(rank=DISENTANGLER_RANK, n_trials=N_TRIALS, shots=SHOTS,
                    calibration_shots=M3_CALIBRATION_SHOTS, optimization_levels=OPT_LEVELS_TO_TEST,
                    dd_policies=DD_POLICIES, correction_methods=list(CORRECTION_METHODS),
                    hardware_rounds=list(CHECKPOINT_ROUNDS), hardware_only=True,
                    tomography_settings=9, density_matrix_fields=["rho_real", "rho_imag"],
                    shared_canonical_routing=False, shared_initial_layout=True, separate_schedules=True,
                    classical_preparation_schedule=_CLASSICAL_PARALLEL_VERSION,
                    classical_checkpoint_construction='independently_scheduled_logical_round_count',
                    logical_qubits=N_LOGICAL_QUBITS, max_rounds=MAX_ROUNDS,
                    notebook_version='NEWv4_CLASSICAL_R45',
                    classical_implementation='NEWv3_overlap_extended_to_five_rounds',
                    mode_block_acquisitions=True,
                    confidence_interval="pointwise two-sided 99% Student-t across trials; n=1 has no CI")
    (OUTPUT_BASE_DIR / "run_configuration.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
    if N_TRIALS < 2:
        print("N_TRIALS=1 retained from the supplied notebooks. No across-trial confidence interval can be estimated. Set N_TRIALS=8 for an eight-trial campaign.")
    for level_index, level in enumerate(OPT_LEVELS_TO_TEST):
        # Only one correction method is enabled; trials are independent.
        modes = CORRECTION_METHODS if level_index % 2 == 0 else tuple(reversed(CORRECTION_METHODS))
        for correction in modes:
            CORRECTION_METHOD = correction
            OPTIMIZATION_LEVEL = int(level)
            OUTPUT_DIR = OUTPUT_BASE_DIR / correction
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            MCM_DD_AUDIT_CSV = OUTPUT_DIR / "dynamic_dd_schedule_audit.csv"
            state = mode_state[correction]
            _SHARED_RECORDS_BY_LEVEL, _SHARED_PLANNED_CONDITIONS_BY_LEVEL = state["records"], state["planned"]
            _SHARED_FAILURES, _SHARED_RECOVERY_EVENTS, _SHARED_AUDIT_ROWS = state["failures"], state["events"], state["audit"]
            try:
                records = run_classical_checkpoint_level_sweep()
            except Exception as exc:
                _shared_failure("optimization_level", exc, correction_method=correction)
                records = _SHARED_RECORDS_BY_LEVEL.get(level, [])
            frame = _shared_records_frame(records)
            frame["optimization_level"] = int(level)
            frame["correction_method"] = correction
            frame.to_csv(OUTPUT_DIR / f"opt_level_{level}_records.csv", index=False)
            if not frame.empty:
                acquired.append(frame)
            combined = pd.concat(acquired, ignore_index=True) if acquired else _shared_empty_records()
            combined.to_csv(OUTPUT_BASE_DIR / "all_correction_methods_combined_records.csv", index=False)
    combined = pd.concat(acquired, ignore_index=True) if acquired else _shared_empty_records()
    combined = _shared_route_comparison(combined)
    for correction, state in mode_state.items():
        CORRECTION_METHOD = correction
        OUTPUT_DIR = OUTPUT_BASE_DIR / correction
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        _SHARED_FAILURES, _SHARED_RECOVERY_EVENTS = state["failures"], state["events"]
        subset = combined[combined["correction_method"].eq(correction)].copy()
        subset.to_csv(OUTPUT_DIR / "all_levels_records.csv", index=False)
        for level in OPT_LEVELS_TO_TEST:
            subset[subset["optimization_level"].eq(level)].to_csv(OUTPUT_DIR / f"opt_level_{level}_records.csv", index=False)
        _shared_update_status(subset, sum(state["planned"].values()))
    combined.to_csv(OUTPUT_BASE_DIR / "all_correction_methods_combined_records.csv", index=False)
    print("Saved correction-method comparison records:", OUTPUT_BASE_DIR)
    return combined


## Run hardware acquisition only when explicitly enabled


In [ ]:
if RUN_EXPERIMENT:
    try:
        all_round_records = run_classical_R456_sweep()
    finally:
        close_hardware_sessions()
else:
    all_round_records = _shared_empty_records()
    print("Ready; hardware-only acquisition is disabled. Set RUN_EXPERIMENT=True intentionally.")
hardware_round_records = all_round_records.copy()

## PNG-only bar plots of the acquired checkpoints


In [ ]:
if not all_round_records.empty:
    replot_folder = replot_existing_sweep(all_round_records)
else:
    print("No new records to plot. Load a saved combined CSV to regenerate figures offline.")